# OpenKind — Phase 2D
## Numerical stability, missing-answer decisions, and complete-request cost

**Builds on your saved Phase 2C run `20260917T222948Z`.** The prior archive is read-only. This notebook reuses the pinned Qwen checkpoint and the exported NLI and candidate heads; it does **not** fine-tune Qwen or change candidate-scoring weights.

**Start with a fresh L4 or A100 Colab runtime, grant access to your `HF_TOKEN` secret, then Run all.** Drive is mounted to read the existing archive and save this run's outputs. The notebook is self-contained; the starter ZIP is not required.

### Questions this experiment answers
1. Which execution-shape differences persist under default BF16, math attention, stricter arithmetic settings, and guarded full-FP32 execution? At which decoder layers do representations begin to differ?
2. With real candidate scores frozen, does refitting the global “none” logit, adding candidate-count information, or using a set-conditioned head improve missing-answer handling?
3. What does **one complete dynamic-choice request** cost, including tokenization, all K candidate encodings, rejection, probabilities, and JSON assembly?

No new Qwen results are prefilled. **CPU fixture tests validate the implementation, not model quality or GPU performance.**

## 1 · Scope and experimental controls

| Kept fixed | Changed or measured |
|---|---|
| Qwen3.5-4B-Base revision and original tokenizer contract | Numerical execution settings, one at a time |
| Both exported Phase 2C neural heads and their feature standardization | A small post-scoring “none” model |
| Last non-padding token representation | Candidate counts 2, 4, 8, 16, plus none |
| The same 57 seen / 20 held-out banking labels | Fresh message splits and lexical hard distractors |

**Three separate experiments:** numerical diagnostics do not silently select a different scoring backend. Rejection fitting always uses the original BF16/default-attention, single-example, unpadded reference. Complete-request batching reports output differences against that same reference.

**Not implemented:** shared-prefix cache forks, a new foundation architecture, LoRA, quantization, generic out-of-domain detection, ordinal Score training, Rust/Metal, or Jev HTTP conformance. The current candidate scorer still encodes the message once **per candidate**.

**Precision caveat:** FP32 here means widening the stored checkpoint values and executing the model in FP32; it cannot recover precision absent from the checkpoint. “Strict” means the listed arithmetic flags, not a promise of universal determinism. Math SDPA affects the full-attention path, not every DeltaNet operation.

## 2 · Install the known-working package family

Keep Colab's CUDA-enabled PyTorch; do not install a replacement torch wheel. The source run used `torch 2.11.0+cu128` and `transformers 5.17.0`. Other PyTorch versions are recorded with a warning; Transformers is pinned for this diagnostic.

The installation cell checks already-imported libraries. If it requests a session restart, do that once and Run all again. Never combine a partially imported old Transformers module with freshly installed files.

In [1]:
import sys, subprocess, importlib.metadata as metadata
from pathlib import Path

PINS = {
    "transformers": "5.17.0", "accelerate": "1.14.0", "datasets": "4.8.5",
    "safetensors": "0.8.0", "numpy": "2.1.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
}
IMPORT_NAMES = {"scikit-learn": "sklearn"}
def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None
restart_after = [name for name, version in PINS.items()
                 if installed_version(name) != version and IMPORT_NAMES.get(name, name) in sys.modules]
requirements = [f"{name}=={version}" for name, version in PINS.items()]
requirements += ["pandas>=2.2,<3", "matplotlib>=3.8,<4", "tqdm>=4.66,<5"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
if restart_after:
    raise RuntimeError("Installed versions changed for imported libraries: " + ", ".join(restart_after)
                       + ". Restart the Colab session, then Run all again.")
print("Dependencies ready. Colab's PyTorch installation was not replaced.")

Dependencies ready. Colab's PyTorch installation was not replaced.


## 3 · Write the auditable experiment modules

These four cells contain the implementation directly. They write only the new `phase2d_*.py` files into the notebook's working directory, not your old notebooks or result archives. Expand a cell to inspect its source. The final reports record source hashes.

Archive loading accepts only bounded, allowlisted JSON, NPZ, and safetensors files. It does not execute code from the previous result ZIP.

In [2]:
%%writefile phase2d_common.py
"""Phase 2D common contracts. Prompt, head, and metric definitions retained from Phase 2C.
No model download or training on import. Imported archives are data, never Python code.
"""
from __future__ import annotations
import copy, gc, hashlib, importlib.metadata as im, json, math, os, random, re, shutil, time, zipfile
from collections import Counter
from contextlib import contextmanager, nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy.optimize import minimize, minimize_scalar
from scipy.special import logsumexp, expit
from sklearn.metrics import f1_score, roc_auc_score
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm
VERSION = "2d.1.0"


LABELS = ["entailment", "neutral", "contradiction"]

NLI_PREFIX = "Decide the relationship between the premise and hypothesis.\nPremise:\n"

NLI_HYP_PREFIX = "\nHypothesis:\n"

NLI_TAIL = ("\nChoices:\nA: entailment. The premise makes the hypothesis true."
            "\nB: neutral. The premise does not establish whether the hypothesis is true or false."
            "\nC: contradiction. The premise makes the hypothesis false."
            "\nReturn the choice code.\nAnswer:")

DYNAMIC_INSTRUCTION = "Select the banking support intent that best describes the customer's message."

DYNAMIC_PREFIX = "Evaluate how well the candidate intent matches the message.\nInstruction:\n"

DYNAMIC_STATE_PREFIX = "\nMessage:\n"

DYNAMIC_CAND_PREFIX = "\nCandidate intent:\n"

DYNAMIC_TAIL = "\nMatch assessment:"

NONE_ID = "__none_of_these__"

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (set, frozenset)):
        return [json_ready(x) for x in sorted(value, key=str)]
    if isinstance(value, (list, tuple)):
        return [json_ready(x) for x in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, (Path, torch.dtype, torch.device)):
        return str(value)
    return value

def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(json_ready(value), indent=2, allow_nan=False)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(payload, encoding="utf-8")
    temp.replace(path)

def sanitize_error(error):
    text = re.sub(r"hf_[A-Za-z0-9_]+", "[REDACTED]", str(error))
    return {"type": type(error).__name__, "message": text[:700]}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalized_text(text):
    return " ".join(text.split()).casefold()

def text_hash(text):
    return hashlib.sha256(normalized_text(text).encode("utf-8")).hexdigest()

def content_hash(value):
    return hashlib.sha256(json.dumps(json_ready(value), sort_keys=True, allow_nan=False).encode()).hexdigest()

def require_finite(tensor, context):
    if not torch.isfinite(tensor).all():
        raise FloatingPointError(f"Non-finite values in {context}; stop rather than replacing them.")

def sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)

def gpu_memory(device):
    if torch.device(device).type != "cuda":
        return {"allocated_mib": None, "reserved_mib": None, "peak_allocated_mib": None}
    return {"allocated_mib": torch.cuda.memory_allocated(device)/1024**2,
            "reserved_mib": torch.cuda.memory_reserved(device)/1024**2,
            "peak_allocated_mib": torch.cuda.max_memory_allocated(device)/1024**2}

def normalize_nli(row, split, index):
    premise, hypothesis = row["premise"].strip(), row["hypothesis"].strip()
    label = int(row["label"])
    if not premise or not hypothesis or label not in range(3):
        return None
    return dict(premise=premise, hypothesis=hypothesis, label=label,
                genre=row.get("genre", "unknown"), group=text_hash(premise),
                pair_hash=text_hash(premise + "\n<HYPOTHESIS>\n" + hypothesis),
                source_split=split, source_index=int(index), pair_id=str(row.get("pairID", index)))

def recover_rows(raw, manifest_rows, n):
    rows = []
    if n > len(manifest_rows):
        raise ValueError("Replication may not add training examples beyond the saved Phase 2B manifest")
    for entry in manifest_rows[:n]:
        row = normalize_nli(raw[entry["source_split"]][entry["source_index"]],
                            entry["source_split"], entry["source_index"])
        if row is None or row["pair_hash"] != entry["pair_hash"] or row["label"] != entry["label"]:
            raise ValueError("Pinned dataset no longer matches the saved source indices/hashes")
        rows.append(row)
    return rows

def encode_nli(row, tokenizer, max_length=256):
    enc = lambda text: tokenizer.encode(text, add_special_tokens=False)
    prefix, hp, tail = enc(NLI_PREFIX), enc(NLI_HYP_PREFIX), enc(NLI_TAIL)
    p, h = enc(row["premise"]), enc(row["hypothesis"])
    budget = max_length-len(prefix)-len(hp)-len(tail)
    if budget < 32:
        raise ValueError("NLI prompt leaves too little content budget")
    hk = h[:max(1, min(len(h), budget-min(24, budget//3)))]
    pk = p[:budget-len(hk)]
    return dict(row, input_ids=prefix+pk+hp+hk+tail,
                premise_truncated=len(pk)<len(p), hypothesis_truncated=len(hk)<len(h))

def pack_tokens(rows, pad_id, device="cpu", pad_to=None, side="right", explicit_positions=False):
    seqs = [r["input_ids"] if isinstance(r, dict) else r for r in rows]
    if not seqs or any(len(x) == 0 for x in seqs):
        raise ValueError("Every input must have at least one non-padding token")
    length = max(map(len, seqs)) if pad_to is None else int(pad_to)
    if length < max(map(len, seqs)) or side not in ("left", "right"):
        raise ValueError("Invalid padding length/side")
    ids = torch.full((len(seqs), length), int(pad_id), dtype=torch.long, device=device)
    mask = torch.zeros_like(ids)
    for i, seq in enumerate(seqs):
        sl = slice(0, len(seq)) if side == "right" else slice(length-len(seq), length)
        ids[i, sl] = torch.tensor(seq, dtype=torch.long, device=device)
        mask[i, sl] = 1
    output = {"input_ids": ids, "attention_mask": mask}
    if explicit_positions:
        output["position_ids"] = (mask.cumsum(-1)-1).clamp_min(0)
    return output

def last_nonpad(hidden, mask):
    if hidden.ndim != 3 or mask.shape != hidden.shape[:2] or not (mask.sum(-1)>0).all():
        raise ValueError("Invalid hidden/mask shapes or all-padding row")
    indices = torch.arange(mask.shape[1], device=mask.device).expand_as(mask)
    last = indices.masked_fill(mask == 0, -1).max(-1).values
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), last]

@torch.inference_mode()
def forward_features(backbone, batch):
    hidden = backbone(**batch, use_cache=False, output_hidden_states=False, return_dict=True).last_hidden_state
    result = last_nonpad(hidden, batch["attention_mask"]).float()
    require_finite(result, "last non-padding hidden representation")
    return result

class FixedHead(nn.Module):
    """State-dict compatible with Phase 2B's last-token linear and MLP heads."""
    def __init__(self, hidden, classes=3, architecture="linear", width=256):
        super().__init__()
        self.architecture = architecture
        self.register_buffer("feature_mean", torch.zeros(hidden))
        self.register_buffer("feature_std", torch.ones(hidden))
        if architecture == "linear":
            self.net = nn.Linear(hidden, classes)
        elif architecture == "mlp":
            self.net = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, width), nn.GELU(),
                                     nn.Dropout(0.1), nn.Linear(width, classes))
        else:
            raise ValueError(architecture)
    def forward(self, x):
        return self.net((x.float()-self.feature_mean)/self.feature_std)
    def fit_normalization(self, x):
        x = torch.as_tensor(x, dtype=torch.float32)
        self.feature_mean.copy_(x.mean(0))
        self.feature_std.copy_(x.std(0).clamp_min(1e-5))

def cpu_state(model):
    return {k: v.detach().cpu().clone().contiguous() for k, v in model.state_dict().items()}

def probs(logits, temperature=1.):
    x = np.asarray(logits, dtype=np.float64)
    if x.ndim != 2 or x.shape[1] < 2 or not np.isfinite(x).all() or not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("Expected finite [N,K>=2] logits and a positive finite temperature")
    z = x/temperature
    return np.exp(z-logsumexp(z, axis=1, keepdims=True))

def probability_metrics(p, y, bins=15):
    p, y = np.asarray(p, dtype=np.float64), np.asarray(y, dtype=np.int64)
    if p.ndim != 2 or len(p)!=len(y) or len(y)==0 or not np.isfinite(p).all():
        raise ValueError("Invalid probability arrays")
    if np.any(p<0) or not np.allclose(p.sum(1),1,atol=1e-6) or np.any(y<0) or np.any(y>=p.shape[1]):
        raise ValueError("Probability normalization/label check failed")
    pred = p.argmax(1)
    conf = p.max(1)
    correct = pred == y
    ece = 0.
    reliability = []
    bucket = np.minimum((conf*bins).astype(int), bins-1)
    for b in range(bins):
        ix = bucket == b
        if ix.any():
            accuracy, confidence = float(correct[ix].mean()), float(conf[ix].mean())
            ece += ix.mean()*abs(accuracy-confidence)
            reliability.append(dict(bin=b, n=int(ix.sum()), accuracy=accuracy, confidence=confidence))
    m = {"n": len(y), "accuracy": float(correct.mean()),
         "macro_f1": float(f1_score(y,pred,labels=list(range(p.shape[1])),average="macro",zero_division=0)),
         "nll": float(-np.log(np.maximum(p[np.arange(len(y)),y], np.finfo(float).tiny)).mean()),
         "brier_sum_classes": float(np.square(p-np.eye(p.shape[1])[y]).sum(1).mean()),
         "ece_toplabel_15bins": float(ece)}
    return m, reliability

def metrics(logits, y, temperature=1.):
    return probability_metrics(probs(logits, temperature), y)[0]

def fit_temperature(logits, labels):
    x, y = np.asarray(logits,dtype=np.float64), np.asarray(labels,dtype=np.int64)
    # Direct log-softmax NLL: no probability clipping in fitting.
    objective = lambda log_t: float(np.mean(logsumexp(x/np.exp(log_t),axis=1)-(x/np.exp(log_t))[np.arange(len(y)),y]))
    opt = minimize_scalar(objective,bounds=(math.log(.25),math.log(4.)),method="bounded")
    if not opt.success or not np.isfinite(opt.fun):
        raise RuntimeError("Temperature optimizer failed")
    return float(np.exp(opt.x))

def calibration_gate(fit_logits, fit_y, gate_logits, gate_y, minimum_improvement=.002):
    fitted = fit_temperature(fit_logits, fit_y)
    raw = metrics(gate_logits, gate_y)
    scaled = metrics(gate_logits, gate_y, fitted)
    improvement = raw["nll"]-scaled["nll"]
    selected = fitted if improvement >= minimum_improvement else 1.
    return {"fitted_temperature": fitted, "selected_temperature": selected,
            "selected_on": "separate calibration_gate NLL, never test data",
            "gate_nll_improvement": improvement, "minimum_required_improvement": minimum_improvement,
            "gate_raw": raw, "gate_temperature_scaled": scaled,
            "temperature_scaling_selected": selected != 1.}

def group_bootstrap(values, groups, repeats=500, seed=17):
    values = np.asarray(values,dtype=np.float64)
    groups = np.asarray(groups)
    unique, inv = np.unique(groups,return_inverse=True)
    totals = np.bincount(inv,weights=values)
    counts = np.bincount(inv)
    if len(unique)<2:
        return {"estimate":float(values.mean()), "ci_low":None,"ci_high":None,"groups":len(unique)}
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(repeats):
        ids = rng.integers(0,len(unique),len(unique))
        estimates.append(totals[ids].sum()/counts[ids].sum())
    low, high = np.quantile(estimates,[.025,.975])
    return dict(estimate=float(values.mean()),ci_low=float(low),ci_high=float(high),groups=len(unique),
                method="95% group bootstrap; sampling uncertainty, not pretraining or training-seed uncertainty")

class CandidateHead(nn.Module):
    """One shared scalar function per (state,instruction,candidate), plus learned null logit.

    No class-specific output matrix and no arbitrary candidate key enters the model.
    The none outcome is a learned baseline, NOT a proven out-of-distribution detector.
    """
    def __init__(self,hidden):
        super().__init__()
        self.scorer=FixedHead(hidden,1,"linear")
        self.none_logit=nn.Parameter(torch.zeros(()))
    def forward(self,x,mask):
        score=self.scorer(x).squeeze(-1).masked_fill(~mask,-1e9)
        return torch.cat([score,self.none_logit.expand(len(x),1)],dim=1)

def parse_banking_csv(content, names):
    import csv
    import io
    mapping = {name: i for i, name in enumerate(names)}
    reader = csv.reader(io.StringIO(content.decode("utf-8-sig")), skipinitialspace=True)
    next(reader)  # original author's file has a two-column header
    rows = []
    for row in reader:
        if len(row) != 2 or row[1] not in mapping:
            raise ValueError("Unexpected Banking77 CSV row or class name")
        rows.append({"text": row[0], "label": mapping[row[1]]})
    return rows

def load_banking_without_scripts(dataset_id, revision, token, cache_root):
    """Read original-author CSVs with SHA-256 checks, never execute legacy dataset scripts.

    The pinned Hugging Face dataset_infos.json supplies class order, URLs and
    expected content hashes. HTTP requests to GitHub never contain HF_TOKEN.
    """
    from huggingface_hub import hf_hub_download
    from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
    from urllib.request import Request, urlopen
    metadata_path = hf_hub_download(dataset_id, "dataset_infos.json", repo_type="dataset",
                                    revision=revision, token=token)
    info = json.loads(Path(metadata_path).read_text())["default"]
    names = info["features"]["label"]["names"]
    if len(names) != 77:
        raise ValueError("Expected the author's 77-label manifest")
    features = Features({"text": Value("string"), "label": ClassLabel(names=names)})
    cache_root = Path(cache_root)/"banking77_csv"
    cache_root.mkdir(parents=True, exist_ok=True)
    datasets, sources = {}, {}
    for split in ("train", "test"):
        urls = [url for url in info["download_checksums"] if url.endswith(f"/banking_data/{split}.csv")]
        if len(urls) != 1:
            raise ValueError("Cannot uniquely resolve original Banking77 CSV from pinned metadata")
        url = urls[0]
        if not url.startswith("https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"):
            raise ValueError("Unexpected Banking77 data origin; inspect metadata rather than executing it")
        expected = info["download_checksums"][url]["checksum"]
        path = cache_root/f"{expected}.csv"
        if path.exists():
            content = path.read_bytes()
        else:
            request = Request(url, headers={"User-Agent": "OpenKind-Phase2D/1.0"})
            with urlopen(request, timeout=90) as response:
                content = response.read(10*1024**2+1)
            if len(content) > 10*1024**2:
                raise ValueError("Banking77 CSV exceeds the bounded download size")
        actual = hashlib.sha256(content).hexdigest()
        if actual != expected:
            raise ValueError(f"Banking77 {split} content hash changed; do not silently use a different dataset")
        if not path.exists():path.write_bytes(content)
        rows = parse_banking_csv(content, names)
        expected_n = int(info["splits"][split]["num_examples"])
        if len(rows) != expected_n:
            raise ValueError(f"Banking77 {split} row count mismatch")
        datasets[split] = Dataset.from_list(rows, features=features)
        sources[split] = {"url": url, "sha256": actual, "examples": len(rows)}
    return DatasetDict(datasets), {"hf_metadata_revision": revision, "loader": "verified original CSVs; no dataset script",
                                  "csv_sources": sources, "license": info.get("license")}

def normalize_banking(row,source,index):
    text=row["text"].strip()
    if not text:
        return None
    return dict(state=text,label=int(row["label"]),group=text_hash(text),source_split=source,source_index=int(index))

def choose_banking(raw,source,n,classes,blocked,seed):
    rows=[]
    for idx in np.random.default_rng(seed).permutation(len(raw[source])):
        row=normalize_banking(raw[source][int(idx)],source,int(idx))
        if row is None or row["label"] not in classes or row["group"] in blocked:
            continue
        rows.append(row); blocked.add(row["group"])
        if len(rows)==n:
            return rows
    raise ValueError(f"Insufficient unique Banking77 messages for {source}: {len(rows)}/{n}")

def encode_candidate(ep,choice,tokenizer,max_length,instruction_override=None):
    enc=lambda s:tokenizer.encode(s,add_special_tokens=False)
    instruction=instruction_override or ep["instruction"]
    pre=enc(DYNAMIC_PREFIX)+enc(instruction)+enc(DYNAMIC_STATE_PREFIX)
    post=enc(DYNAMIC_CAND_PREFIX)+enc(choice["description"])+enc(DYNAMIC_TAIL)
    budget=max_length-len(pre)-len(post)
    if budget<16:
        raise ValueError("Instruction/candidate text too long; never silently truncate the candidate definition")
    state=enc(ep["state"])
    return {"input_ids":pre+state[:budget]+post,"state_truncated":len(state)>budget,
            "group":ep["group"]}

def pad_episode_logits(logits,labels):
    width=max(map(len,logits))
    x=np.full((len(logits),width),-1e9,dtype=np.float64)
    y=[]
    for i,(row,label) in enumerate(zip(logits,labels)):
        k=len(row)-1
        x[i,:k]=row[:k];x[i,-1]=row[-1]
        y.append(width-1 if label==k else label)
    return x,np.asarray(y,dtype=np.int64)

def episode_metrics(logits,episodes,temperature=1.):
    labels=[e["target_index"] for e in episodes]
    matrix,targets=pad_episode_logits(logits,labels)
    m=metrics(matrix,targets,temperature)
    m.pop("macro_f1")  # position-specific macro-F1 is not meaningful for dynamic labels
    p=[probs(np.asarray(row)[None,:],temperature)[0] for row in logits]
    pred=np.array([r.argmax() for r in p])
    correct=pred==np.array(labels)
    absent=np.array([e["target_index"]==len(e["choices"]) for e in episodes])
    abstain=np.array([pred[i]==len(e["choices"]) for i,e in enumerate(episodes)])
    m.update(coverage=float((~abstain).mean()),none_target_rate=float(absent.mean()),
             none_recall=float(abstain[absent].mean()) if absent.any() else None,
             false_abstention_rate=float(abstain[~absent].mean()) if (~absent).any() else None,
             answered_accuracy=float(correct[~abstain].mean()) if (~abstain).any() else None,
             answerable_accuracy=float(correct[~absent].mean()) if (~absent).any() else None,
             none_auroc=float(roc_auc_score(absent,[r[-1] for r in p])) if len(np.unique(absent))==2 else None)
    return m

@dataclass
class Settings:
    preset: str = 'standard'  # smoke / quick / standard; smoke is plumbing only
    prior_archive: str = '/content/drive/MyDrive/Colab Notebooks/OpenKind_Phase2C_results/20260917T222948Z/openkind_phase2c_20260917T222948Z.zip'
    expected_archive_sha256: str = '2952c2fcca6638c0cc1d1e1060a567b849e46424eb2d16a078d1945233bffb63'
    model_id: str = 'Qwen/Qwen3.5-4B-Base'
    model_revision: str = '1001bb4d826a52d1f399e183466143f4da7b741b'
    nli_revision: str = 'da70db2af9d09693783c3320c4249840212ee221'
    banking_revision: str = '90d4e2ee5521c04fc1488f065b8b083658768c57'
    output_root: str = '/content/openkind_phase2d'
    resume_run_id: str = ''  # same-runtime resume only; must match saved config and source hash
    mount_drive: bool = True
    save_to_drive: bool = True
    run_numerics: bool = True
    run_fp32: bool = True   # one model converted temporarily, with a memory guard
    run_none_ablation: bool = True
    run_request_benchmark: bool = True
    diagnostic_seed: int = 17041
    data_seed: int = 17043
    max_length: int = 256
    fit_candidate_counts: tuple = (2,4,8,16)
    eval_candidate_counts: tuple = (2,4,8,16)
    samplers: tuple = ('uniform', 'label_lexical_hard')
    benchmark_batches: tuple = (1,2,4)
    benchmark_repeats: int = 5
    benchmark_warmups: int = 1
    probability_tolerance: float = 0.005
    thresholds: tuple = (0.5,0.8,0.9,0.95)
    none_l2: float = 0.001
    calibration_gate_minimum: float = 0.002
    bootstrap_repeats: int = 500
    score_cache_commit_every: int = 50

    def sizes(self):
        return {
            'smoke': dict(numerics=2, trace=1, train=12,dev=6,cal_fit=6,cal_gate=6,test_seen=4,test_unseen=4,benchmark_messages=1),
            'quick': dict(numerics=8,trace=2,train=120,dev=40,cal_fit=60,cal_gate=40,test_seen=24,test_unseen=24,benchmark_messages=2),
            'standard': dict(numerics=32,trace=4,train=500,dev=150,cal_fit=200,cal_gate=150,test_seen=100,test_unseen=100,benchmark_messages=4),
        }[self.preset]

    def validate(self):
        if self.preset not in ('smoke','quick','standard'): raise ValueError('Unknown preset')
        if not (128 <= self.max_length <= 512): raise ValueError('max_length must be 128..512')
        if not self.fit_candidate_counts or not self.eval_candidate_counts: raise ValueError('Empty candidate counts')
        if min(self.fit_candidate_counts+self.eval_candidate_counts)<2: raise ValueError('At least two real candidates')
        if max(self.fit_candidate_counts+self.eval_candidate_counts)>19: raise ValueError('20-label holdout requires K<=19 for omitted-answer episodes')
        if set(self.samplers)-{'uniform','label_lexical_hard'}: raise ValueError('Unknown sampler')
        if not self.samplers or len(set(self.samplers))!=len(self.samplers): raise ValueError('Unique nonempty samplers required')
        if min(self.benchmark_batches)<1 or self.benchmark_repeats<1 or self.benchmark_warmups<0: raise ValueError('Invalid benchmarking settings')
        if self.none_l2<0 or self.bootstrap_repeats<2 or self.score_cache_commit_every<1: raise ValueError('Invalid optimization/cache setting')
        if self.resume_run_id and not re.fullmatch(r'[A-Za-z0-9_-]+',self.resume_run_id): raise ValueError('Invalid resume ID')
        if not re.fullmatch(r'[0-9a-f]{64}',self.expected_archive_sha256): raise ValueError('Require pinned reference archive SHA256')


def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()


def read_reference(cfg, destination):
    """Allowlisted bounded extraction; no pickle or archived source-code execution."""
    archive=Path(cfg.prior_archive)
    if not archive.is_file(): raise FileNotFoundError(f'Mount Drive or set prior_archive: {archive}')
    sha=file_sha(archive)
    if sha!=cfg.expected_archive_sha256: raise ValueError('Reference ZIP checksum mismatch. Use the specified Phase 2C run, not an arbitrary archive.')
    names=['paste_back_summary.json','nli_split_manifest.json','dynamic_episodes.json','dynamic_data_audit.json','stability.json',
           'frozen_nli_export/manifest.json','frozen_nli_export/head.safetensors','frozen_nli_export/golden_head_inputs.npz','frozen_nli_export/golden_token_inputs.json',
           'dynamic_export/manifest.json','dynamic_export/candidate_head.safetensors','dynamic_export/golden_candidate_inputs.npz']
    root=Path(destination);root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if len(z.namelist())!=len(set(z.namelist())):raise ValueError('Duplicate ZIP members')
        for name in names:
            info=z.getinfo(name)
            if info.file_size>40*1024**2:raise ValueError('Oversized reference member')
            p=root/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(info))
    ref={'root':root,'sha256':sha}
    for key,name in [('summary','paste_back_summary.json'),('nli_manifest','nli_split_manifest.json'),('episodes','dynamic_episodes.json'),('data','dynamic_data_audit.json'),('stability','stability.json'),('nli_export','frozen_nli_export/manifest.json'),('dynamic_export','dynamic_export/manifest.json')]:
        ref[key]=json.loads((root/name).read_text())
    if ref['summary']['run_id']!='20260917T222948Z':raise ValueError('Wrong reference run')
    for m in [ref['nli_export'],ref['dynamic_export']]:
        if m['model_revision']!=cfg.model_revision or m['model_id']!=cfg.model_id:raise ValueError('Model reference mismatch')
    if ref['nli_export']['label_order']!=LABELS:raise ValueError('NLI class order mismatch')
    if cfg.max_length!=ref['dynamic_export']['max_length']:raise ValueError('Keep the trained prompt max_length for Phase 2D')
    return ref


def load_heads_and_check(ref):
    h=FixedHead(ref['nli_export']['hidden_size']).eval()
    h.load_state_dict(load_file(str(ref['root']/'frozen_nli_export/head.safetensors')),strict=True)
    c=CandidateHead(ref['dynamic_export']['hidden_size']).eval()
    c.load_state_dict(load_file(str(ref['root']/'dynamic_export/candidate_head.safetensors')),strict=True)
    for m in (h,c):
        for p in m.parameters():p.requires_grad_(False)
    with torch.no_grad():
        with np.load(ref['root']/'frozen_nli_export/golden_head_inputs.npz',allow_pickle=False) as z:
            got=h(torch.from_numpy(z['features'])).numpy();expected=z['raw_logits']
            hd=float(np.max(np.abs(got-expected)))
        with np.load(ref['root']/'dynamic_export/golden_candidate_inputs.npz',allow_pickle=False) as z:
            got=c(torch.from_numpy(z['features']),torch.from_numpy(z['candidate_mask'])).numpy();expected=z['logits']
            cd=float(np.max(np.abs(got-expected)))
    if hd>1e-4 or cd>1e-4:raise ValueError(f'Exported head fixtures do not reproduce: NLI={hd}, candidate={cd}')
    for head in [h,c.scorer]:
        if not torch.isfinite(head.feature_std).all() or not (head.feature_std>0).all():raise ValueError('Invalid normalization buffers')
    return h,c,{'nli_max_abs_logit_difference':hd,'candidate_max_abs_logit_difference':cd,
                'scope':'CPU head fixture parity only, not GPU backbone parity','frozen':True}

Writing phase2d_common.py


In [3]:
%%writefile phase2d_numerics.py
"""Paired execution-shape diagnostics. No training, no KV reuse, no kernel monkey-patches."""
from phase2d_common import *

class DiagnosticSkip(RuntimeError):
    pass

class IntegrityError(RuntimeError):
    """Never continue after a diagnostic altered the model samples."""
    pass


def backend_flags():
    values={'float32_matmul_precision':torch.get_float32_matmul_precision()}
    for path,obj,keys in [('cuda.matmul',torch.backends.cuda.matmul,('allow_tf32','allow_bf16_reduced_precision_reduction','allow_fp16_reduced_precision_reduction')),
                          ('cudnn',torch.backends.cudnn,('allow_tf32','benchmark','deterministic'))]:
        for k in keys:
            if hasattr(obj,k):values[path+'.'+k]=getattr(obj,k)
    return values


def integer_sample_indices(numel,device):
    # Float32 linspace can round numel-1 up to numel for giant embedding matrices.
    if numel<1:raise ValueError('Cannot sample an empty tensor')
    count=min(16,numel)
    return torch.arange(count,device=device,dtype=torch.int64)*(numel-1)//max(count-1,1)


def parameter_sample_hash(model):
    """Cheap alteration detector, not a full weight checksum; model revision pins full weights."""
    h=hashlib.sha256()
    for name,t in list(model.named_parameters())+list(model.named_buffers()):
        if t.numel()==0:continue
        flat=t.detach().reshape(-1)
        indices=integer_sample_indices(t.numel(),t.device)
        h.update(name.encode());h.update(flat[indices].float().cpu().numpy().tobytes())
    return h.hexdigest()


@contextmanager
def execution_mode(model,mode,device):
    """Temporarily change numerical settings; restore per-tensor dtypes and global flags."""
    allowed=('bf16_default','bf16_math','bf16_strict_math','fp32_strict_math')
    if mode not in allowed:raise ValueError(mode)
    original=backend_flags()
    tensors=list(model.parameters())+list(model.buffers())
    original_dtypes={('parameter',n):t.dtype for n,t in model.named_parameters()}
    original_dtypes.update({('buffer',n):t.dtype for n,t in model.named_buffers()})
    initial_hash=parameter_sample_hash(model)
    wants_fp32=mode.startswith('fp32')
    if wants_fp32 and torch.device(device).type=='cuda':
        gc.collect();torch.cuda.empty_cache();sync(device)
        free,_=torch.cuda.mem_get_info(device)
        delta=sum(max(0,t.numel()*4-t.numel()*t.element_size()) for t in tensors if t.is_floating_point())
        largest=max((t.numel()*4 for t in tensors if t.is_floating_point()),default=0)
        required=delta+largest+2*1024**3
        if free<required:
            raise DiagnosticSkip(f'FP32 guard: {free/1024**3:.2f} GiB free; conservative conversion/workspace requirement {required/1024**3:.2f} GiB. No silent CPU offload.')
    try:
        if wants_fp32:model.to(dtype=torch.float32)
        del tensors
        if 'strict' in mode:
            torch.set_float32_matmul_precision('highest')
            for obj,key,val in [(torch.backends.cuda.matmul,'allow_tf32',False),
                                (torch.backends.cuda.matmul,'allow_bf16_reduced_precision_reduction',False),
                                (torch.backends.cuda.matmul,'allow_fp16_reduced_precision_reduction',False),
                                (torch.backends.cudnn,'allow_tf32',False),
                                (torch.backends.cudnn,'benchmark',False),
                                (torch.backends.cudnn,'deterministic',True)]:
                if hasattr(obj,key):setattr(obj,key,val)
        ctx=nullcontext()
        if mode!='bf16_default':
            from torch.nn.attention import sdpa_kernel, SDPBackend
            ctx=sdpa_kernel(SDPBackend.MATH)
        with ctx:
            model.eval()
            yield {'name':mode,'parameter_dtypes':dict(Counter(str(t.dtype) for t in model.parameters())),
                   'settings':backend_flags(),'forced_sdpa_math':mode!='bf16_default',
                   'scope':'full model FP32 only in fp32_strict_math; math SDPA alone changes full-attention kernels, not DeltaNet kernels'}
    finally:
        # BF16 parameters are exactly representable in FP32. Restore original dtypes,
        # including originally-FP32 buffers instead of blindly casting every buffer to BF16.
        with torch.no_grad():
            current={('parameter',n):t for n,t in model.named_parameters()}
            current.update({('buffer',n):t for n,t in model.named_buffers()})
            for key,dt in original_dtypes.items():
                t=current[key]
                if t.dtype!=dt:t.data=t.data.to(dtype=dt)
        torch.set_float32_matmul_precision(original['float32_matmul_precision'])
        for prefix,obj in [('cuda.matmul',torch.backends.cuda.matmul),('cudnn',torch.backends.cudnn)]:
            for key in ('allow_tf32','allow_bf16_reduced_precision_reduction','allow_fp16_reduced_precision_reduction','benchmark','deterministic'):
                if prefix+'.'+key in original:setattr(obj,key,original[prefix+'.'+key])
        gc.collect()
        if torch.device(device).type=='cuda':torch.cuda.empty_cache();sync(device)
        if parameter_sample_hash(model)!=initial_hash:raise IntegrityError('Parameter/buffer samples changed during a diagnostic; stop before training or scoring.')


@torch.inference_mode()
def one_forward(model,head,batch,trace=False):
    """Capture only the target row's last valid token, not all layer/token tensors."""
    snapshots={};handles=[]
    if trace:
        index=int(torch.nonzero(batch['attention_mask'][0],as_tuple=False)[-1].item())
        modules={'embedding':model.get_input_embeddings()}
        modules.update({f'layer_{i:02d}':layer for i,layer in enumerate(model.layers)})
        modules['final_norm']=model.norm
        def make_hook(name):
            def hook(_module,_args,out):
                out=out[0] if isinstance(out,tuple) else out
                if not isinstance(out,torch.Tensor) or out.ndim!=3:raise RuntimeError(f'Unexpected trace output {name}')
                snapshots[name]=out[0,index].detach().float().cpu().numpy().copy()
            return hook
        for name,module in modules.items():handles.append(module.register_forward_hook(make_hook(name)))
    try:
        feature=forward_features(model,batch)
        logits=head(feature).float()
        require_finite(logits,'numerical diagnostic logits')
        p=torch.softmax(logits,dim=-1).cpu().numpy()[0]
        return p,feature.cpu().numpy()[0],snapshots
    finally:
        for handle in handles:handle.remove()


def numerical_rows(raw,ref,tokenizer,cfg):
    """Reuse known development rows for debugging; do not treat them as fresh test data."""
    manifest=ref['nli_manifest']['dev']
    rows=recover_rows(raw,manifest,len(manifest))
    by_group={r['group']:r for r in rows}
    scores={}
    for r in ref['stability']['records']:
        if r['group'] in by_group:scores[r['group']]=max(scores.get(r['group'],0),r['max_probability_difference'])
    n=cfg.sizes()['numerics'];stress_n=min(max(1,n//4),len(scores))
    stress=sorted(scores,key=lambda g:(-scores[g],g))[:stress_n]
    rest=sorted(set(by_group)-set(stress));rng=np.random.default_rng(cfg.diagnostic_seed);rng.shuffle(rest)
    chosen=stress+rest[:max(0,n-len(stress))]
    result=[]
    for g in chosen:
        r=encode_nli(by_group[g],tokenizer,cfg.max_length)
        r['selection']='prior_high_drift' if g in stress else 'random_dev'
        result.append(r)
    return result,{'n':len(result),'prior_high_drift':len(stress),'random_dev':len(result)-len(stress),
                   'scope':'purposive debugging sample; not a population estimate or fresh quality benchmark',
                   'groups':[r['group'] for r in result]}


def scenarios(row,short,long):
    n=len(row['input_ids'])
    return [('repeat_alone',[row],{}),('duplicate_batch2',[row,row],{}),
            ('right_pad32',[row],{'pad_to':n+32}),('right_pad128',[row],{'pad_to':n+128}),
            ('mixed_batch3',[row,short,long],{}),
            ('left_pad32_explicit',[row],{'pad_to':n+32,'side':'left','explicit_positions':True})]


def summarize_shapes(records,cfg):
    result=[]
    for mode,scenario in sorted({(r['mode'],r['scenario']) for r in records}):
        rs=[r for r in records if r['mode']==mode and r['scenario']==scenario]
        ds=np.array([r['max_probability_delta'] for r in rs])
        result.append({'mode':mode,'scenario':scenario,'n':len(rs),'p50_delta':float(np.median(ds)),
                       'p95_delta':float(np.quantile(ds,.95)),'max_delta':float(ds.max()),
                       'class_flips':sum(r['class_flip'] for r in rs),
                       'examples_over_tolerance':int((ds>cfg.probability_tolerance).sum()),
                       'threshold_crossings':{str(t):sum(r['threshold_crossings'][str(t)] for r in rs) for t in cfg.thresholds}})
    return result


def run_numerics(model,head,rows,tokenizer,cfg,device,out):
    head=head.to(device).eval();out=Path(out);out.mkdir(parents=True,exist_ok=True)
    modes=['bf16_default','bf16_math','bf16_strict_math']+(['fp32_strict_math'] if cfg.run_fp32 else [])
    records=[];layer_rows=[];mode_status={};single={};baseline_single={}
    short=min(rows,key=lambda r:len(r['input_ids']));long=max(rows,key=lambda r:len(r['input_ids']))
    for mode in modes:
        print('Numerical diagnostic:',mode)
        try:
            with execution_mode(model,mode,device) as metadata:
                began=time.perf_counter()
                for i,row in enumerate(tqdm(rows,desc=mode)):
                    traced=i<cfg.sizes()['trace']
                    base,h0,t0=one_forward(model,head,pack_tokens([row],tokenizer.pad_token_id,device),traced)
                    single[(mode,row['group'])]=base.tolist()
                    if mode=='bf16_default':baseline_single[row['group']]=base
                    for name,seqs,kw in scenarios(row,short,long):
                        p,h1,t1=one_forward(model,head,pack_tokens(seqs,tokenizer.pad_token_id,device,**kw),traced)
                        delta=float(np.max(np.abs(p-base)))
                        rec={'mode':mode,'scenario':name,'group':row['group'],'selection':row['selection'],
                             'tokens':len(row['input_ids']),'max_probability_delta':delta,
                             'class_flip':bool(p.argmax()!=base.argmax()),'single_probabilities':base.tolist(),'alternate_probabilities':p.tolist(),
                             'hidden_relative_l2':float(np.linalg.norm(h1-h0)/max(float(np.linalg.norm(h0)),1e-12)),
                             'threshold_crossings':{str(t):bool(np.any((base>=t)!=(p>=t))) for t in cfg.thresholds}}
                        records.append(rec)
                        for layer in t0:
                            a,b=t0[layer],t1[layer]
                            layer_rows.append({'mode':mode,'scenario':name,'group':row['group'],'layer':layer,
                                               'layer_type':model.config.layer_types[int(layer.split('_')[1])] if layer.startswith('layer_') else layer,
                                               'relative_l2':float(np.linalg.norm(a-b)/max(float(np.linalg.norm(a)),1e-12)),
                                               'max_abs':float(np.max(np.abs(a-b))),'mean_abs':float(np.mean(np.abs(a-b)))})
                mode_status[mode]={'status':'completed','seconds':time.perf_counter()-began,'configuration':metadata}
        except IntegrityError:
            raise
        except (DiagnosticSkip,torch.OutOfMemoryError) as exc:
            mode_status[mode]={'status':'skipped_memory' if isinstance(exc,DiagnosticSkip) else 'failed_oom','error':sanitize_error(exc)}
        except Exception as exc:
            mode_status[mode]={'status':'failed','error':sanitize_error(exc)}
            if mode=='bf16_default':
                json_write(out/'numerics_partial.json',{'modes':mode_status,'records':records})
                raise
        json_write(out/'numerics_partial.json',{'modes':mode_status,'records':records})
    if not cfg.run_fp32:mode_status['fp32_strict_math']={'status':'disabled'}
    summaries=summarize_shapes(records,cfg)
    drift=[]
    for (mode,g),p in single.items():
        if mode!='bf16_default' and g in baseline_single:
            base=baseline_single[g];p=np.asarray(p)
            drift.append({'mode':mode,'group':g,'max_probability_delta':float(np.max(np.abs(p-base))),
                          'class_flip':bool(p.argmax()!=base.argmax())})
    import pandas as pd
    pd.DataFrame(layer_rows).to_csv(out/'layer_drift.csv',index=False)
    json_write(out/'shape_records.json',records)
    json_write(out/'single_precision_drift.json',drift)
    verdict={}
    for mode,info in mode_status.items():
        rs=[r for r in summaries if r['mode']==mode]
        verdict[mode]=('within_sampled_tolerance' if rs and all(r['class_flips']==0 and r['examples_over_tolerance']==0 for r in rs)
                       else 'needs_review') if info['status']=='completed' else info['status']
    report={'modes':mode_status,'shape_summary':summaries,'verdict':verdict,'tolerance':cfg.probability_tolerance,
            'layer_trace_rows':len(layer_rows),'traced_examples_per_mode':min(len(rows),cfg.sizes()['trace']),
            'trace_file':'numerics/layer_drift.csv','no_trained_parameters_changed':True,
            'single_precision_shift':{mode:{'max_delta':max(r['max_probability_delta'] for r in drift if r['mode']==mode),
                                          'class_flips':sum(r['class_flip'] for r in drift if r['mode']==mode)} for mode in set(r['mode'] for r in drift)},
            'scope':'within-mode paired shape comparison plus between-mode single-call drift; not accuracy, not exact reproducibility across hardware',
            'note':'No numerical mode is automatically selected for the subsequent scoring stage. BF16 default single-example execution is retained.'}
    json_write(out/'numerics_summary.json',report)
    head.to('cpu')
    return report

Writing phase2d_numerics.py


In [4]:
%%writefile phase2d_decisions.py
"""Frozen candidate scoring, controlled missing-answer ablations, complete-request timing."""
from phase2d_common import *
import sqlite3

FEATURE_NAMES=['max_score','top_two_gap','mean_score','std_score','logmeanexp','log_K']


def overlap_audit(splits,blocked):
    groups={s:{r['group'] for r in rows} for s,rows in splits.items()}
    for s,g in groups.items():
        if g & blocked:raise AssertionError(f'Phase 2C message overlap in {s}')
    comparisons={}
    for i,(a,ga) in enumerate(groups.items()):
        for b,gb in list(groups.items())[i+1:]:
            comparisons[a+'__'+b]=len(ga & gb)
            if ga & gb:raise AssertionError('New phase data overlap')
    return comparisons


def make_episode(row,names,candidate_ids,missing,k,sampler):
    ids=list(map(int,candidate_ids))
    choices=[{'id':f'intent_{i}','description':names[i].replace('_',' '),'source_label':i} for i in ids]
    target=len(ids) if missing else ids.index(row['label'])
    return dict(id=f"{row['source_split']}_{row['source_index']}_{sampler}_k{k}_abs{int(missing)}",
                state=row['state'],instruction=DYNAMIC_INSTRUCTION,choices=choices,
                target_index=target,target=NONE_ID if missing else choices[target]['id'],
                group=row['group'],source_split=row['source_split'],source_index=row['source_index'],
                source_label=row['label'],real_candidate_count=k,true_intent_omitted=bool(missing),sampler=sampler)


def build_fresh_episodes(raw,ref,cfg):
    """Disjoint new messages; same 57/20 label partition; nested distractors across K."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    names=list(raw['train'].features['label'].names)
    if names!=ref['data']['class_names']:raise ValueError('Banking label order changed')
    seen=ref['data']['training_label_ids'];unseen=ref['data']['heldout_label_ids']
    if set(seen)&set(unseen) or len(seen)!=57 or len(unseen)!=20:raise ValueError('Unexpected reference label split')
    blocked={ep['group'] for es in ref['episodes'].values() for ep in es}
    # Lexical difficulty uses only the label descriptions, not learned test-message features.
    # The true label is used OFFLINE to choose plausible distractors; it is not a production retriever.
    similarity=np.zeros((len(names),len(names)),dtype=np.float64)
    # Separate vocabularies prevent unseen descriptions influencing fit-time IDF weights.
    for label_pool in (seen,unseen):
        vectors=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5)).fit_transform([names[i].replace('_',' ') for i in label_pool])
        similarity[np.ix_(label_pool,label_pool)]=(vectors@vectors.T).toarray()
    all_used=set(blocked);message_splits={};episodes={}
    for j,s in enumerate(('train','dev','cal_fit','cal_gate','test_seen','test_unseen')):
        pool=unseen if s=='test_unseen' else seen
        source='test' if s.startswith('test') else 'train'
        rows=choose_banking(raw,source,cfg.sizes()[s],set(pool),all_used,cfg.data_seed+j)
        message_splits[s]=rows;eps=[]
        for row in rows:
            seed=int(row['group'][:12],16)+cfg.data_seed
            rng=np.random.default_rng(seed)
            negatives=[c for c in pool if c!=row['label']]
            uniform=list(rng.permutation(negatives))
            # Deterministic random tie-breaks; never rely on integer class position.
            tie={int(c):float(rng.random()) for c in negatives}
            hard=sorted(negatives,key=lambda c:(-similarity[row['label'],c],tie[c]))
            orders={'uniform':uniform,'label_lexical_hard':hard}
            ks=cfg.eval_candidate_counts if s.startswith('test') else (int(rng.choice(cfg.fit_candidate_counts)),)
            samplers=cfg.samplers if s.startswith('test') else (str(rng.choice(cfg.samplers)),)
            for sampler in samplers:
                for k in ks:
                    for missing in (False,True):
                        ids=orders[sampler][:k] if missing else [row['label']]+orders[sampler][:k-1]
                        if len(ids)!=k:raise ValueError('Not enough negative categories')
                        order_rng=np.random.default_rng(seed+k*101+int(missing)*37+(1 if sampler=='uniform' else 2))
                        ids=list(order_rng.permutation(ids))
                        eps.append(make_episode(row,names,ids,missing,k,sampler))
        episodes[s]=eps
    audit=overlap_audit(message_splits,blocked)
    for s in ('train','dev','cal_fit','cal_gate','test_seen'):
        if any(c['source_label'] in unseen for ep in episodes[s] for c in ep['choices']):raise AssertionError('Held-out label leaked')
    if any(c['source_label'] not in unseen for ep in episodes['test_unseen'] for c in ep['choices']):raise AssertionError('Incorrect unseen pool')
    return episodes,{'prior_messages_excluded':len(blocked),'overlaps':audit,
                     'label_holdout_scope':'same 20 labels excluded from Phase 2C scorer training and Phase 2D rejection fitting/calibration, not Qwen pretraining',
                     'label_ids_seen':seen,'label_ids_unseen':unseen,
                     'splits':{s:{'messages':len(rows),'episodes':len(episodes[s]),'true_answer_absent_fraction':0.5,
                                  'candidate_counts':dict(Counter(len(e['choices']) for e in episodes[s])),
                                  'samplers':dict(Counter(e['sampler'] for e in episodes[s]))} for s,rows in message_splits.items()},
                     'construction':'paired present/absent episodes; nested negative lists across K within message and sampler',
                     'fit_dev_calibration_absent_prior':0.25,
                     'hard_negative_definition':'character 3..5 gram TF-IDF similarity between true-label and negative-label descriptions; random tie-break. Not model-mined hard negatives.',
                     'scope':'one domain and sampled candidates; evaluation labels are unseen only to head stages; not a 77-way benchmark or universal OOD detection'}


@dataclass
class ScoredEpisodes:
    episodes: list
    scores: list

    def subset(self,indices):return ScoredEpisodes([self.episodes[i] for i in indices],[self.scores[i] for i in indices])


def cache_identity(cfg,ref,environment):
    return {'version':VERSION,'model':cfg.model_id,'revision':cfg.model_revision,
            'candidate_head_sha256':file_sha(ref['root']/'dynamic_export/candidate_head.safetensors'),
            'dtype':'torch.bfloat16','batch':1,'backend':'default_sdpa','environment':environment,
            'max_length':cfg.max_length,'prompt_contract':ref['dynamic_export']['prompt_segments']}


@torch.inference_mode()
def score_episodes(episodes,model,candidate_head,tokenizer,cfg,cache_path,identity,device):
    """Cache scalar logits, not all token features. One unpadded prompt per model call.
    SQLite transactions survive an interrupted cell; never reuse other precision/backend fingerprints.
    """
    unique={};references=[];trunc=0
    for ep in episodes:
        keys=[]
        for c in ep['choices']:
            row=encode_candidate(ep,c,tokenizer,cfg.max_length)
            key=content_hash({'identity':identity,'tokens':row['input_ids']})
            keys.append(key)
            if key not in unique:
                unique[key]=row;trunc+=int(row['state_truncated'])
        references.append(keys)
    Path(cache_path).parent.mkdir(parents=True,exist_ok=True)
    connection=sqlite3.connect(str(cache_path));connection.execute('PRAGMA journal_mode=WAL')
    connection.execute('CREATE TABLE IF NOT EXISTS scores (key TEXT PRIMARY KEY, score REAL NOT NULL)')
    values={};misses=[]
    for key in unique:
        entry=connection.execute('SELECT score FROM scores WHERE key=?',(key,)).fetchone()
        if entry is None:misses.append(key)
        elif math.isfinite(entry[0]):values[key]=float(entry[0])
        else:raise ValueError('Nonfinite cached score')
    candidate_head.to(device).eval();began=time.perf_counter()
    try:
        for i,key in enumerate(tqdm(misses,desc='Unpadded frozen candidate scores')):
            row=unique[key]
            batch=pack_tokens([row],tokenizer.pad_token_id,device)
            h=forward_features(model,batch)
            s=candidate_head.scorer(h).reshape(-1)
            require_finite(s,'frozen candidate score')
            value=float(s.item());values[key]=value
            connection.execute('INSERT INTO scores(key,score) VALUES (?,?)',(key,value))
            if (i+1)%cfg.score_cache_commit_every==0:connection.commit()
        connection.commit()
    finally:
        connection.commit();connection.close()
        candidate_head.to('cpu')
    scores=[np.array([values[k] for k in ks],dtype=np.float64) for ks in references]
    return ScoredEpisodes(episodes,scores),{'candidate_appearances':sum(map(len,references)),'unique_prompts':len(unique),
        'cache_hits':len(unique)-len(misses),'computed':len(misses),'seconds':time.perf_counter()-began,
        'unique_truncated_prompts':trunc,'cache_identity':content_hash(identity),'feature_batch_size':1,
        'state_reencoded_per_candidate':True}


def score_features(scores,kind):
    if kind in ('frozen_global','refit_global'):return np.empty((len(scores),0),dtype=np.float64)
    if kind=='count_bias':return np.array([[np.log(len(s))] for s in scores],dtype=np.float64)
    if kind!='set_linear':raise ValueError(kind)
    result=[]
    for s in scores:
        s=np.asarray(s,dtype=np.float64)
        if len(s)<2 or not np.isfinite(s).all():raise ValueError('Need finite >=2 candidate scores')
        top=np.sort(s)[-2:]
        result.append([top[-1],top[-1]-top[-2],s.mean(),s.std(),logsumexp(s)-np.log(len(s)),np.log(len(s))])
    return np.asarray(result)


def prior_weights(episodes,prior=0.25):
    """Equal message weight; within each message the explicit absent mass is prior.
    This reweights a controlled paired stress set. It is NOT an observed deployment prior.
    """
    if not 0<prior<1:raise ValueError('Prior must lie strictly between zero and one')
    counts=Counter((e['group'],bool(e['true_intent_omitted'])) for e in episodes)
    weights=[]
    for e in episodes:
        g=e['group'];absent=bool(e['true_intent_omitted'])
        if (g,True) not in counts or (g,False) not in counts:raise ValueError('Both paired conditions required per message')
        weights.append((prior if absent else 1-prior)/counts[g,absent])
    w=np.asarray(weights);return w/w.sum()


@dataclass
class NoneModel:
    kind: str
    coef: list
    feature_mean: list
    feature_std: list

    def none_logits(self,scores):
        f=score_features(scores,self.kind)
        if f.shape[1]:f=(f-np.asarray(self.feature_mean))/np.asarray(self.feature_std)
        x=np.column_stack([np.ones(len(scores)),f])
        z=x@np.asarray(self.coef)
        if not np.isfinite(z).all():raise FloatingPointError('Nonfinite none scores')
        return z

    def logits(self,scores):
        return [np.r_[s,b] for s,b in zip(scores,self.none_logits(scores))]


def per_episode_nll(logits,episodes,temperature=1.):
    if not math.isfinite(temperature) or temperature<=0:raise ValueError('Invalid temperature')
    return np.array([logsumexp(np.asarray(s)/temperature)-s[e['target_index']]/temperature for s,e in zip(logits,episodes)])


def weighted_nll(logits,episodes,prior=.25,temperature=1.):
    return float(prior_weights(episodes,prior)@per_episode_nll(logits,episodes,temperature))


def fit_none_model(train,kind,initial_none,l2=.001):
    f=score_features(train.scores,kind)
    mean=f.mean(0);std=np.maximum(f.std(0),1e-5)
    x=np.column_stack([np.ones(len(f)),(f-mean)/std])
    weights=prior_weights(train.episodes)
    absent=np.array([e['true_intent_omitted'] for e in train.episodes],dtype=np.float64)
    lse=np.array([logsumexp(s) for s in train.scores])
    true_score=np.array([0. if e['true_intent_omitted'] else s[e['target_index']] for s,e in zip(train.scores,train.episodes)])
    def objective(theta):
        b=x@theta
        loss=np.logaddexp(lse,b)-np.where(absent>0,b,true_score)
        penalty=0.5*l2*np.dot(theta[1:],theta[1:])
        grad=x.T@(weights*(expit(b-lse)-absent));grad[1:]+=l2*theta[1:]
        return float(weights@loss+penalty),grad
    initial=np.r_[initial_none,np.zeros(f.shape[1])]
    opt=minimize(objective,initial,jac=True,method='L-BFGS-B',options={'maxiter':1000,'ftol':1e-12,'gtol':1e-8})
    if not opt.success or not np.isfinite(opt.fun):raise RuntimeError(f'None optimizer failed: {opt.message}')
    model=NoneModel(kind,opt.x.tolist(),mean.tolist(),std.tolist())
    return model,{'optimizer':'L-BFGS-B, analytic gradient','success':bool(opt.success),'iterations':int(opt.nit),
                  'parameters':len(opt.x),'training_weighted_nll':weighted_nll(model.logits(train.scores),train.episodes),
                  'l2_non_intercept':l2,'candidate_scorer_changed':False,'assumed_absent_prior':.25}


def extended_metrics(logits,episodes,temperature=1.):
    result=episode_metrics(logits,episodes,temperature)
    absent=np.array([e['true_intent_omitted'] for e in episodes])
    pred=np.array([int(np.argmax(s)) for s in logits])
    k=np.array([len(e['choices']) for e in episodes]);reject=pred==k
    correct=pred==np.array([e['target_index'] for e in episodes])
    result.update({'absent_n':int(absent.sum()),'present_n':int((~absent).sum()),
                   'false_answers_when_absent':int((absent&~reject).sum()),
                   'false_answer_rate_when_absent':float((~reject)[absent].mean()) if absent.any() else None,
                   'false_abstentions_when_present':int((~absent&reject).sum()),
                   'wrong_real_choice_when_present':int((~absent&~reject&~correct).sum()),
                   'messages':len({e['group'] for e in episodes})})
    return result


def select_rejection_models(train,dev,candidate_head,cfg):
    initial=float(candidate_head.none_logit.detach().cpu())
    models={'frozen_global':NoneModel('frozen_global',[initial],[],[])}
    fits={'frozen_global':{'status':'unchanged_phase2c','parameters_trained':0}}
    for kind in ('refit_global','count_bias','set_linear'):
        models[kind],fits[kind]=fit_none_model(train,kind,initial,cfg.none_l2)
    comparison={}
    for name,model in models.items():
        logits=model.logits(dev.scores)
        comparison[name]={'weighted_nll_prior025':weighted_nll(logits,dev.episodes),
                          'paired_stress_metrics':extended_metrics(logits,dev.episodes)}
    winner=min(comparison,key=lambda k:(comparison[k]['weighted_nll_prior025'],len(models[k].coef),k))
    return models,{'selected':winner,'criterion':'lowest dev message-weighted NLL at assumed omitted-intent prior 0.25; ties prefer fewer parameters',
                   'development':comparison,'training':fits,'frozen_candidate_scores':True,
                   'scope':'selected head is not a validated safety policy; held-out errors and tradeoffs must still be inspected'}


def weighted_temperature_gate(model,fit,gate,minimum=.002):
    fit_z=model.logits(fit.scores);gate_z=model.logits(gate.scores)
    objective=lambda t:weighted_nll(fit_z,fit.episodes,temperature=float(np.exp(t)))
    opt=minimize_scalar(objective,bounds=(np.log(.25),np.log(4.)),method='bounded')
    if not opt.success or not np.isfinite(opt.fun):raise RuntimeError('Temperature fitting failed')
    t=float(np.exp(opt.x));raw=weighted_nll(gate_z,gate.episodes);scaled=weighted_nll(gate_z,gate.episodes,temperature=t)
    return {'fitted_temperature':t,'selected_temperature':t if raw-scaled>=minimum else 1.,
            'gate_raw_weighted_nll':raw,'gate_scaled_weighted_nll':scaled,'improvement':raw-scaled,
            'minimum_improvement':minimum,'assumed_absent_prior':.25,'selected_on':'separate calibration-gate messages, never test',
            'temperature_changes_argmax':False}


def evaluate_rejection(store,models,selected,temperature,cfg):
    report={};predictions={}
    for name,model in models.items():
        logits=model.logits(store.scores);t=temperature if name==selected else 1.
        predictions[name]=[{'episode_id':e['id'],'group':e['group'],'sampler':e['sampler'],'K':len(e['choices']),
                            'absent':e['true_intent_omitted'],'target_index':e['target_index'],
                            'logits':s.tolist(),'probabilities':probs(s[None,:],t)[0].tolist(),
                            'prediction_index':int(s.argmax())} for e,s in zip(store.episodes,logits)]
        by={}
        for k in sorted(set(len(e['choices']) for e in store.episodes)):
            for sampler in sorted(set(e['sampler'] for e in store.episodes)):
                ix=[i for i,e in enumerate(store.episodes) if len(e['choices'])==k and e['sampler']==sampler]
                if ix:by[f'k{k}/{sampler}']=extended_metrics([logits[i] for i in ix],[store.episodes[i] for i in ix],t)
        correctness=[int(np.argmax(s)==e['target_index']) for s,e in zip(logits,store.episodes)]
        report[name]={'raw':extended_metrics(logits,store.episodes),
                      'reported_temperature':t,'temperature_scaled':extended_metrics(logits,store.episodes,t),
                      'weighted_nll_prior_scenarios':{str(p):weighted_nll(logits,store.episodes,p,t) for p in (.05,.25,.5)},
                      'by_K_and_sampler':by,'accuracy_interval_message_bootstrap':group_bootstrap(correctness,[e['group'] for e in store.episodes],cfg.bootstrap_repeats,cfg.data_seed)}
    # Paired estimate on the SAME test episodes; messages, not candidate-count copies, are resampled.
    base=models['frozen_global'].logits(store.scores);new=models[selected].logits(store.scores)
    difference=np.array([int(a.argmax()==e['target_index'])-int(b.argmax()==e['target_index']) for a,b,e in zip(new,base,store.episodes)])
    absent=np.array([e['true_intent_omitted'] for e in store.episodes]);groups=np.array([e['group'] for e in store.episodes])
    delta_absent=np.array([int(a.argmax()<len(e['choices']))-int(b.argmax()<len(e['choices'])) for a,b,e in zip(new,base,store.episodes)])
    paired={'selected_minus_frozen_accuracy':group_bootstrap(difference,groups,cfg.bootstrap_repeats,cfg.data_seed),
            'selected_minus_frozen_false_answer_rate_when_absent':group_bootstrap(delta_absent[absent],groups[absent],cfg.bootstrap_repeats,cfg.data_seed)}
    return {'models':report,'selected':selected,'paired':paired,
            'denominator_note':'paired stress set is 50% absent; weighted NLL scenarios are assumptions, not observed deployment priors'},predictions


@torch.inference_mode()
def real_request(ep,model,candidate_head,tokenizer,none_model,cfg,device,batch_size,temperature):
    # Include all tokenization, H2D, K candidate forwards, none/head computation, CPU sync and JSON serialization.
    rows=[encode_candidate(ep,c,tokenizer,cfg.max_length) for c in ep['choices']]
    scores=[];calls=0
    for start in range(0,len(rows),batch_size):
        batch=pack_tokens(rows[start:start+batch_size],tokenizer.pad_token_id,device)
        x=forward_features(model,batch)
        z=candidate_head.scorer(x).reshape(-1)
        require_finite(z,'full request candidate scores')
        scores.extend(z.float().cpu().numpy().tolist());calls+=1
    z=none_model.logits([np.asarray(scores,dtype=np.float64)])[0]
    p=probs(z[None,:],temperature)[0]
    ids=[c['id'] for c in ep['choices']]+[NONE_ID]
    response=json.dumps({'selected':ids[int(p.argmax())],'probabilities':dict(zip(ids,p.tolist()))},allow_nan=False)
    return response,p,{'model_calls':calls,'input_tokens_total':sum(len(r['input_ids']) for r in rows),
                       'candidate_count':len(rows),'max_candidate_tokens':max(len(r['input_ids']) for r in rows),
                       'state_reencoded':len(rows)}


def benchmark_requests(episodes,model,candidate_head,tokenizer,none_model,cfg,device,temperature):
    candidate_head.to(device).eval();rows=[]
    # One small fixed set of DEV messages, reused at every K and candidate batch size.
    messages={}
    for ep in episodes:
        if not ep['true_intent_omitted'] and ep['sampler']=='uniform':messages.setdefault(ep['group'],[]).append(ep)
    selected=list(messages)[:cfg.sizes()['benchmark_messages']]
    chosen=[]
    for g in selected:
        byk={len(e['choices']):e for e in messages[g]}
        chosen.extend(byk[k] for k in cfg.eval_candidate_counts if k in byk)
    if not chosen:raise ValueError('No benchmark episodes')
    try:
        for ep in tqdm(chosen,desc='Complete dynamic requests'):
            _,refp,_=real_request(ep,model,candidate_head,tokenizer,none_model,cfg,device,1,temperature)
            for batch_size in cfg.benchmark_batches:
                try:
                    for _ in range(cfg.benchmark_warmups):real_request(ep,model,candidate_head,tokenizer,none_model,cfg,device,batch_size,temperature)
                    sync(device)
                    baseline=torch.cuda.memory_allocated(device) if torch.device(device).type=='cuda' else None
                    if baseline is not None:torch.cuda.reset_peak_memory_stats(device)
                    times=[];deltas=[];flips=[]
                    for _ in range(cfg.benchmark_repeats):
                        sync(device);t0=time.perf_counter()
                        response,p,info=real_request(ep,model,candidate_head,tokenizer,none_model,cfg,device,batch_size,temperature)
                        sync(device);times.append(1000*(time.perf_counter()-t0))
                        del response
                        deltas.append(float(np.max(np.abs(p-refp))));flips.append(bool(p.argmax()!=refp.argmax()))
                    rows.append(dict(episode_id=ep['id'],group=ep['group'],candidate_batch_size=batch_size,**info,status='completed',
                                     p50_ms=float(np.median(times)),p95_ms=float(np.quantile(times,.95)),times_ms=times,
                                     max_probability_delta_vs_unpadded=max(deltas),selection_changed=any(flips),
                                     requests_per_second=1000/float(np.median(times)),candidates_per_second=len(ep['choices'])*1000/float(np.median(times)),
                                     resident_mib=baseline/1024**2 if baseline is not None else None,
                                     peak_extra_mib=(torch.cuda.max_memory_allocated(device)-baseline)/1024**2 if baseline is not None else None))
                except torch.OutOfMemoryError as exc:
                    rows.append({'episode_id':ep['id'],'candidate_batch_size':batch_size,'status':'failed_oom','error':sanitize_error(exc)})
                    gc.collect();torch.cuda.empty_cache()
    finally:candidate_head.to('cpu')
    return {'rows':rows,'distinct_messages':len(selected),'scope':'complete single Python request, no server/network; real dev messages; no caches of hidden state or logits in timing; K full state encodings',
            'p95_warning':'few repeated calls: descriptive timing quantile, not a production tail-latency guarantee',
            'shared_prefill':False,'generation':False}


def benchmark_episodes(dev_episodes,ref,cfg):
    # Expand ONLY dev messages, with reproducible nested uniform negatives. No test-set timing selection.
    names=ref['data']['class_names'];pool=ref['data']['training_label_ids'];unique={}
    for e in dev_episodes:unique.setdefault(e['group'],e)
    result=[]
    for g,e in list(unique.items())[:cfg.sizes()['benchmark_messages']]:
        row={'state':e['state'],'label':e['source_label'],'group':g,'source_split':e['source_split'],'source_index':e['source_index']}
        rng=np.random.default_rng(int(g[:12],16)+cfg.data_seed)
        negatives=list(rng.permutation([i for i in pool if i!=row['label']]))
        for k in cfg.eval_candidate_counts:
            ids=list(rng.permutation([row['label']]+negatives[:k-1]))
            result.append(make_episode(row,names,ids,False,k,'uniform'))
    return result

Writing phase2d_decisions.py


In [5]:
%%writefile phase2d_workflow.py
"""Colab/CLI orchestration, reproducible reports, and non-destructive Drive output."""
from phase2d_common import *
from phase2d_numerics import *
from phase2d_decisions import *
import platform, sys, inspect
from datetime import datetime, timezone

SOURCES={
 'qwen_api':'https://huggingface.co/docs/transformers/model_doc/qwen3_5',
 'qwen_implementation':'https://github.com/huggingface/transformers/blob/v5.17.0/src/transformers/models/qwen3_5/modeling_qwen3_5.py',
 'pytorch_numerics':'https://docs.pytorch.org/docs/main/notes/numerical_accuracy.html',
 'sdpa_control':'https://docs.pytorch.org/docs/main/generated/torch.nn.attention.sdpa_kernel.html',
 'native_bf16':'https://docs.pytorch.org/docs/main/generated/torch.cuda.is_bf16_supported.html',
 'banking77':'https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data',
}

class Workflow:
    def __init__(self,cfg):
        cfg.validate();self.cfg=cfg
        self.run_id=cfg.resume_run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        self.out=Path(cfg.output_root)/self.run_id;self.out.mkdir(parents=True,exist_ok=True)
        self.cache=Path(cfg.output_root)/'score_cache';self.cache.mkdir(parents=True,exist_ok=True)
        effective=asdict(cfg);effective.pop('resume_run_id')
        config_path=self.out/'configuration.json'
        if config_path.exists() and json.loads(config_path.read_text())!=json_ready(effective):
            raise ValueError('Resume config differs. Use a new run_id; do not overwrite previous results.')
        json_write(config_path,effective)
        self.report={'schema':'openkind-phase2d-summary/v1','run_id':self.run_id,'version':VERSION,
                     'configuration':effective,'stages':{},'warnings':[], 'sources':SOURCES,
                     'limitations':[
                      'Frozen Qwen3.5 and frozen NLI/candidate heads. Only small none heads and optional temperature are fitted.',
                      'The numerical sample deliberately includes old high-drift examples: debugging, not a frequency estimate.',
                      'Fresh dynamic test messages exclude every Phase 2C message; rerunning this data_seed reuses them.',
                      'No Qwen pretraining decontamination or cross-domain validity is established.',
                      'None means omitted annotated banking intent, not a universal unknown/safety detector.',
                      'Lexical hard negatives are label-description based, not model-mined hard negatives.',
                      'Paired tests are 50% absent. Weighted training/selection uses a declared 25% absent prior, not a measured deployment prior.',
                      'The scorer re-encodes the state for each candidate. No KV branching, generation, LoRA, quantization, Rust/Metal or HTTP validation.',
                      'Timing is a bounded single-process experiment; percentile estimates are descriptive, not production latency guarantees.',
                     ]}
        self.device=None;self.token=None;self.models=None;self.selected='frozen_global';self.temperature=1.
        if cfg.preset!='standard':self.report['warnings'].append(f'{cfg.preset}: small sample; smoke results validate execution only.')
        self.checkpoint('initialized')

    def checkpoint(self,stage,status='completed'):
        self.report['stages'][stage]=status
        json_write(self.out/'partial_report.json',self.report)

    @contextmanager
    def stage(self,name):
        self.checkpoint(name,'running')
        try:
            yield
            self.checkpoint(name)
        except Exception as exc:
            self.report.setdefault('errors',{})[name]=sanitize_error(exc)
            self.checkpoint(name,'failed')
            raise

    def setup(self):
        with self.stage('setup'):
            self.token=os.environ.get('HF_TOKEN')
            try:
                from google.colab import userdata
                self.token=userdata.get('HF_TOKEN') or self.token
            except Exception:pass
            if not torch.cuda.is_available():raise RuntimeError('Use a fresh L4 or A100 GPU runtime. Local CPU validation is separate.')
            self.device=torch.device('cuda:0')
            if torch.cuda.memory_allocated(self.device)>512*1024**2:raise RuntimeError('Existing GPU tensors: restart session before loading another model.')
            with torch.cuda.device(self.device):native=torch.cuda.is_bf16_supported(including_emulation=False)
            if not native:raise RuntimeError('Phase 2D requires native BF16 (L4/A100). T4 emulated BF16/FP16 would change this experiment.')
            free,total=torch.cuda.mem_get_info(self.device)
            if free<11*1024**3:raise RuntimeError('Need at least 11 GiB free for the BF16 reference; FP32 has an additional guard.')
            self.dtype=torch.bfloat16
            environment={'python':sys.version.split()[0],'platform':platform.platform(),'gpu':torch.cuda.get_device_name(self.device),
                         'native_bf16':native,'compute_capability':list(torch.cuda.get_device_capability(self.device)),
                         'cuda_runtime':torch.version.cuda,'gpu_total_gib':total/1024**3,'initial_free_gib':free/1024**3,
                         'packages':{p:im.version(p) for p in ('torch','transformers','accelerate','datasets','safetensors','numpy','scipy','scikit-learn')},
                         'backend_flags':backend_flags()}
            optional={}
            for p in ('flash-linear-attention','fla-core','causal-conv1d','flash-attn'):
                try:optional[p]=im.version(p)
                except im.PackageNotFoundError:optional[p]=None
            environment['optional_kernel_packages']=optional
            if environment['packages']['transformers']!='5.17.0':raise RuntimeError('Use transformers==5.17.0 for the reference; restart after installation. Do not silently benchmark another implementation.')
            if not environment['packages']['torch'].startswith('2.11.'):
                self.report['warnings'].append('PyTorch differs from the previous 2.11 run. Record this as a new environment, not a direct software-identical replication.')
            if any(optional.values()):self.report['warnings'].append('Optional kernels are installed. This differs from Phase 2C; package presence alone does not prove which kernel executed.')
            self.report['code_sha256']={n:file_sha(Path(__file__).with_name(n)) for n in ('phase2d_common.py','phase2d_numerics.py','phase2d_decisions.py','phase2d_workflow.py')}
            self.report['environment']=environment
            if self.cfg.mount_drive and self.cfg.prior_archive.startswith('/content/drive/'):
                from google.colab import drive
                drive.mount('/content/drive',force_remount=False)
            self.ref=read_reference(self.cfg,self.out/'source_reference')
            self.nli_head,self.candidate_head,parity=load_heads_and_check(self.ref)
            # Source contract is authoritative: do not silently change prefix text or tokenization.
            expected={'prefix':DYNAMIC_PREFIX,'state_prefix':DYNAMIC_STATE_PREFIX,'candidate_prefix':DYNAMIC_CAND_PREFIX,'tail':DYNAMIC_TAIL}
            if self.ref['dynamic_export']['prompt_segments']!=expected:raise RuntimeError('Dynamic prompt contract mismatch')
            self.report['source']={'run_id':self.ref['summary']['run_id'],'archive_sha256':self.ref['sha256'],
                                   'model_id':self.cfg.model_id,'model_revision':self.cfg.model_revision,'head_fixture_parity':parity}
            json_write(self.out/'environment.json',environment)
            print(json.dumps(json_ready(environment),indent=2));print('Read-only reference:',self.ref['summary']['run_id'])

    def load_model(self):
        with self.stage('model'):
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.tokenizer=AutoTokenizer.from_pretrained(self.cfg.model_id,revision=self.cfg.model_revision,token=self.token,trust_remote_code=False)
            if self.tokenizer.pad_token_id is None:self.tokenizer.pad_token=self.tokenizer.eos_token
            if self.tokenizer.pad_token_id is None:raise RuntimeError('Missing pad token')
            self.tokenizer.padding_side='right'
            print('Loading ONE text-only model at the pinned Phase 2C revision.')
            lm,info=AutoModelForCausalLM.from_pretrained(self.cfg.model_id,revision=self.cfg.model_revision,token=self.token,
                    trust_remote_code=False,dtype=self.dtype,device_map={'':0},attn_implementation='sdpa',output_loading_info=True)
            info=json_ready(info);json_write(self.out/'loading_info.json',info)
            if type(lm).__name__!='Qwen3_5ForCausalLM' or type(lm.model).__name__!='Qwen3_5TextModel':raise RuntimeError('Unexpected model classes')
            if any('visual' in n for n,_ in lm.named_modules()):raise RuntimeError('A vision tower was loaded unexpectedly')
            if any(info.get(k) for k in ('missing_keys','mismatched_keys','error_msgs')):raise RuntimeError('Incomplete model weights; inspect loading_info.json')
            self.backbone=lm.model;del lm
            self.backbone.eval()
            for p in self.backbone.parameters():p.requires_grad_(False)
            gc.collect();torch.cuda.empty_cache()
            implementation=inspect.getfile(type(self.backbone))
            self.report['model']={'class':type(self.backbone).__name__,'parameters':sum(p.numel() for p in self.backbone.parameters()),
                                  'implementation_sha256':file_sha(implementation),'layer_types':list(self.backbone.config.layer_types),
                                  'dtype_counts':dict(Counter(str(p.dtype) for p in self.backbone.parameters())),
                                  'generation':False,'vision':False,'shared_prefill':False,'memory':gpu_memory(self.device)}
            json_write(self.out/'model.json',self.report['model'])
            # Replay the historical small token fixtures, without pretending cross-device exactness.
            fixtures=json.loads((self.ref['root']/'frozen_nli_export/golden_token_inputs.json').read_text())
            self.nli_head.to(self.device)
            h=[]
            for r in fixtures:
                h.append(forward_features(self.backbone,pack_tokens([r],self.tokenizer.pad_token_id,self.device)).cpu().numpy()[0])
            h=np.asarray(h)
            with np.load(self.ref['root']/'frozen_nli_export/golden_head_inputs.npz',allow_pickle=False) as z:
                old=z['features'].copy()
            with torch.no_grad():
                pnew=torch.softmax(self.nli_head(torch.tensor(h,device=self.device)),dim=-1).cpu().numpy()
                pold=torch.softmax(self.nli_head(torch.tensor(old,device=self.device)),dim=-1).cpu().numpy()
            self.nli_head.to('cpu')
            self.report['historical_token_replay']={'n':len(h),'max_feature_difference':float(np.max(np.abs(h-old))),
                 'max_probability_difference':float(np.max(np.abs(pnew-pold))),
                 'class_changes':int((pnew.argmax(1)!=pold.argmax(1)).sum()),'scope':'small source replay, not a fresh accuracy test'}
            print(json.dumps(self.report['historical_token_replay'],indent=2))

    def load_data(self):
        with self.stage('data'):
            if self.cfg.run_numerics:
                from datasets import load_dataset
                required_splits=sorted({r['source_split'] for r in self.ref['nli_manifest']['dev']})
                raw={split:load_dataset('nyu-mll/multi_nli',revision=self.cfg.nli_revision,split=split,token=self.token)
                     for split in required_splits}
                self.numerical_tokens,self.report['numerical_selection']=numerical_rows(raw,self.ref,self.tokenizer,self.cfg)
                del raw;gc.collect()
                json_write(self.out/'numerical_inputs.json',[{k:r[k] for k in ('input_ids','group','selection')} for r in self.numerical_tokens])
            if self.cfg.run_none_ablation or self.cfg.run_request_benchmark:
                raw,sources=load_banking_without_scripts('PolyAI/banking77',self.cfg.banking_revision,self.token,self.cache)
                self.episodes,self.report['dynamic_data']=build_fresh_episodes(raw,self.ref,self.cfg)
                self.report['dynamic_data']['sources']=sources
                del raw;gc.collect()
                json_write(self.out/'episodes.json',self.episodes);json_write(self.out/'data_audit.json',self.report['dynamic_data'])
                import pandas as pd
                print(pd.DataFrame(self.report['dynamic_data']['splits']).T[['messages','episodes']].to_string())
                print('All Phase 2C messages excluded. Present/absent pairs are stress tests; fitting/selection is reweighted to 25% absent.')

    def numerical_diagnostics(self):
        if not self.cfg.run_numerics:self.checkpoint('numerics','disabled');return
        with self.stage('numerics'):
            self.report['numerics']=run_numerics(self.backbone,self.nli_head,self.numerical_tokens,self.tokenizer,self.cfg,self.device,self.out/'numerics')
            print(json.dumps(self.report['numerics']['verdict'],indent=2))

    def score(self):
        if not self.cfg.run_none_ablation:self.checkpoint('scoring','disabled');return
        with self.stage('scoring'):
            from phase2d_numerics import backend_flags
            identity=cache_identity(self.cfg,self.ref,{'packages':self.report['environment']['packages'],'gpu':self.report['environment']['gpu'],
                              'cuda_runtime':self.report['environment'].get('cuda_runtime'),
                              'optional_kernels':self.report['environment'].get('optional_kernel_packages'),
                              'flags_now':backend_flags(),'model_code_sha256':self.report['model']['implementation_sha256'],
                              'experiment_code_sha256':self.report.get('code_sha256',{})})
            self.stores={};self.report['scoring']={}
            for split,eps in self.episodes.items():
                print('Scoring split:',split)
                self.stores[split],meta=score_episodes(eps,self.backbone,self.candidate_head,self.tokenizer,self.cfg,self.cache/'scores.sqlite3',identity,self.device)
                self.report['scoring'][split]=meta
                json_write(self.out/f'scores_{split}.json',{'episode_ids':[e['id'] for e in eps],'scores':self.stores[split].scores})
                self.checkpoint('scoring','running')

    def fit_rejection(self):
        if not self.cfg.run_none_ablation:
            self.models={'frozen_global':NoneModel('frozen_global',[float(self.candidate_head.none_logit.detach())],[],[])}
            self.temperature=float(self.ref['dynamic_export']['selected_temperature'])
            self.checkpoint('none_fit','disabled');return
        with self.stage('none_fit'):
            self.models,self.report['selection']=select_rejection_models(self.stores['train'],self.stores['dev'],self.candidate_head,self.cfg)
            self.selected=self.report['selection']['selected']
            self.report['temperature']=weighted_temperature_gate(self.models[self.selected],self.stores['cal_fit'],self.stores['cal_gate'],self.cfg.calibration_gate_minimum)
            self.temperature=self.report['temperature']['selected_temperature']
            json_write(self.out/'none_models.json',{k:asdict(v) for k,v in self.models.items()})
            json_write(self.out/'selection.json',self.report['selection']);json_write(self.out/'temperature_gate.json',self.report['temperature'])
            print('Selected on development NLL:',self.selected,'; temperature:',self.temperature)

    def evaluate(self):
        if not self.cfg.run_none_ablation:self.checkpoint('evaluation','disabled');return
        with self.stage('evaluation'):
            self.report['evaluation']={}
            for split in ('test_seen','test_unseen'):
                result,predictions=evaluate_rejection(self.stores[split],self.models,self.selected,self.temperature,self.cfg)
                self.report['evaluation'][split]=result
                json_write(self.out/f'predictions_{split}.json',predictions)
                print(split,json.dumps(result['models'][self.selected]['temperature_scaled'],indent=2))
            json_write(self.out/'evaluation.json',self.report['evaluation'])

    def benchmark(self):
        if not self.cfg.run_request_benchmark:self.checkpoint('request_benchmark','disabled');return
        with self.stage('request_benchmark'):
            eps=benchmark_episodes(self.episodes['dev'],self.ref,self.cfg)
            self.report['request_benchmark']=benchmark_requests(eps,self.backbone,self.candidate_head,self.tokenizer,self.models[self.selected],self.cfg,self.device,self.temperature)
            json_write(self.out/'request_benchmark.json',self.report['request_benchmark'])
            import pandas as pd
            df=pd.DataFrame([r for r in self.report['request_benchmark']['rows'] if r['status']=='completed'])
            if len(df):print(df.groupby(['candidate_count','candidate_batch_size'])[['p50_ms','max_probability_delta_vs_unpadded']].median().to_string())

    def export(self):
        with self.stage('export'):
            dest=self.out/'export';dest.mkdir(exist_ok=True)
            shutil.copy2(self.ref['root']/'dynamic_export/candidate_head.safetensors',dest/'frozen_candidate_head.safetensors')
            for k,m in self.models.items():json_write(dest/f'none_{k}.json',asdict(m))
            manifest={'format':'openkind-frozen-scorer-none-head/v1','model_id':self.cfg.model_id,'model_revision':self.cfg.model_revision,
                      'source_archive_sha256':self.ref['sha256'],'candidate_head_sha256':file_sha(dest/'frozen_candidate_head.safetensors'),
                      'candidate_contract':self.ref['dynamic_export'],'selected_none_head':self.selected,'none_model':asdict(self.models[self.selected]),
                      'none_feature_names':FEATURE_NAMES if self.selected=='set_linear' else (['log_K'] if self.selected=='count_bias' else []),
                      'feature_definition':'max, max-second_max, arithmetic mean, population std(ddof=0), logsumexp(s)-log(K), natural log(K); evaluated in float64',
                      'temperature':self.temperature,'probability_arithmetic':'float64 log-softmax after appending none logit; temperature applies to all K+1 logits',
                      'fit_absent_prior_assumption':.25,'base_weights_included':False,'batch_invariant':False,
                      'scope':'Python none/scoring export; Rust/Metal and Jev HTTP untested; no shared prefill'}
            json_write(dest/'manifest.json',manifest)
            fixture=[]
            # Developer fixtures are from DEV scores, never fabricated benchmark results.
            if hasattr(self,'stores'):
                for e,s in zip(self.stores['dev'].episodes[:8],self.stores['dev'].scores[:8]):
                    z=self.models[self.selected].logits([s])[0]
                    fixture.append({'id':e['id'],'candidate_scores':s.tolist(),'none_logit':float(z[-1]),
                                    'all_logits':z.tolist(),'probabilities':probs(z[None,:],self.temperature)[0].tolist()})
                json_write(dest/'golden_none_inputs.json',fixture)
            self.report['export']={'manifest':'export/manifest.json','candidate_scorer_unchanged':True,'none_fixture_count':len(fixture),
                                   'validation':'fixture values from Python; not a Rust runtime validation'}

    def finalize(self):
        self.checkpoint('report','running')
        make_plots(self.report,self.out)
        self.checkpoint('report')
        json_write(self.out/'openkind_phase2d_summary.json',self.report)
        summary=compact_summary(self.report)
        json_write(self.out/'paste_back_summary.json',summary)
        readout=(f"# OpenKind Phase 2D\n\nRun: {self.run_id}\n\n"
                 f"Source: {self.report.get('source',{}).get('run_id')}\n\n"
                 f"Numerical verdicts: {self.report.get('numerics',{}).get('verdict',{})}\n\n"
                 f"Development-selected none model: {self.selected}\n\n"
                 "## Interpretation boundaries\n"+'\n'.join('- '+x for x in self.report['limitations']))
        (self.out/'openkind_phase2d_summary.md').write_text(readout)
        archive=Path(self.cfg.output_root)/f'openkind_phase2d_{self.run_id}.zip'
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            for p in sorted(self.out.rglob('*')):
                if p.is_file() and 'source_reference' not in p.parts:z.write(p,p.relative_to(self.out))
        print('Local results:',self.out);print('Result archive:',archive)
        drive_status='disabled'
        if self.cfg.save_to_drive:
            try:
                from google.colab import drive
                drive.mount('/content/drive',force_remount=False)
                dest=Path('/content/drive/MyDrive/Colab Notebooks/OpenKind_Phase2D_results')/self.run_id
                dest.mkdir(parents=True,exist_ok=True)
                for p in [archive,self.out/'paste_back_summary.json',self.out/'openkind_phase2d_summary.json',self.out/'openkind_phase2d_summary.md']:
                    shutil.copy2(p,dest/p.name)
                drive_status='copied';print('Copied reports/archive to:',dest)
            except Exception as exc:
                drive_status='failed';print('Drive copy failed; local files are preserved:',sanitize_error(exc))
        json_write(self.out/'delivery_status.json',{'drive':drive_status,'archive':str(archive)})
        print('===== BEGIN_OPENKIND_PHASE2D_SUMMARY =====')
        print(json.dumps(json_ready(summary),indent=2,allow_nan=False))
        print('===== END_OPENKIND_PHASE2D_SUMMARY =====')
        return archive


def compact_summary(report):
    summary={k:report.get(k) for k in ('schema','run_id','version','source','environment','code_sha256','stages','warnings','limitations')}
    summary['numerics']=report.get('numerics',{'status':'not_run'})
    summary['data']=report.get('dynamic_data',{})
    if 'sources' in summary['data']:summary['data']=dict(summary['data']);summary['data'].pop('sources',None)
    summary['none_selection']=report.get('selection',{})
    summary['temperature']=report.get('temperature',{})
    summary['evaluation']={}
    selected=report.get('selection',{}).get('selected','frozen_global')
    for split,result in report.get('evaluation',{}).items():
        summary['evaluation'][split]={'selected':selected,'paired':result['paired'],
             'models':{name:result['models'][name] for name in dict.fromkeys(['frozen_global',selected])}}
    rows=report.get('request_benchmark',{}).get('rows',[])
    summary['request_benchmark']={'scope':report.get('request_benchmark',{}).get('scope'),'rows':[
        {k:r[k] for k in ('episode_id','candidate_count','candidate_batch_size','status','p50_ms','p95_ms','model_calls','input_tokens_total','max_probability_delta_vs_unpadded','selection_changed','peak_extra_mib') if k in r} for r in rows]}
    summary['export']=report.get('export',{})
    summary['readiness']='research_only; inspect shape drift, false-answer/false-abstention tradeoff, and evidence scope before deployment'
    return summary


def make_plots(report,out):
    import matplotlib.pyplot as plt
    import pandas as pd
    root=Path(out)/'plots';root.mkdir(exist_ok=True)
    title_prefix='SYNTHETIC VALIDATION — NOT QWEN RESULTS\n' if report.get('validation_only') else ''
    shapes=report.get('numerics',{}).get('shape_summary',[])
    if shapes:
        df=pd.DataFrame(shapes);scenarios_=list(dict.fromkeys(df.scenario));modes=list(dict.fromkeys(df['mode']))
        fig,ax=plt.subplots(figsize=(11,5));x=np.arange(len(scenarios_));w=.8/max(1,len(modes))
        for j,mode in enumerate(modes):
            vals=[100*float(df[(df['mode']==mode)&(df.scenario==s)].p95_delta.iloc[0]) if len(df[(df['mode']==mode)&(df.scenario==s)]) else np.nan for s in scenarios_]
            ax.bar(x-.4+w/2+j*w,vals,w,label=mode)
        ax.set_xticks(x,[s.replace('_','\n') for s in scenarios_]);ax.set_ylabel('95th percentile max |Δ probability|\n(percentage points)')
        ax.set_title(title_prefix+'Execution-shape differences within each numerical configuration');ax.legend(fontsize=8)
        fig.tight_layout();fig.savefig(root/'precision_shape_drift.png',dpi=150);plt.close(fig)
    path=Path(out)/'numerics/layer_drift.csv'
    if path.exists() and path.stat().st_size>10:
        df=pd.read_csv(path);df=df[(df.scenario=='duplicate_batch2')&df.layer.str.startswith('layer_')].copy()
        if len(df):
            df['layer_index']=df.layer.str.split('_').str[-1].astype(int)
            fig,ax=plt.subplots(figsize=(9,5))
            for mode,part in df.groupby('mode'):
                a=part.groupby('layer_index').relative_l2.mean();ax.plot(a.index,a.values,marker='.',label=mode)
            ax.set_xlabel('Decoder layer index (0-based)');ax.set_ylabel('Mean relative L2 difference at last valid token')
            ax.set_title(title_prefix+'Where duplicate-batch representations diverge — traced sample');ax.legend(fontsize=8)
            fig.tight_layout();fig.savefig(root/'layer_drift.png',dpi=150);plt.close(fig)
    result=report.get('evaluation',{}).get('test_unseen')
    if result:
        fig,ax=plt.subplots(figsize=(9,5))
        for name in dict.fromkeys(['frozen_global',result['selected']]):
            by=result['models'][name]['by_K_and_sampler'];pairs=[]
            for key,m in by.items():
                if key.endswith('/label_lexical_hard'):pairs.append((int(key.split('/')[0][1:]),100*m['false_answer_rate_when_absent']))
            pairs.sort()
            if pairs:ax.plot([a for a,b in pairs],[b for a,b in pairs],marker='o',label=name)
        ax.set_xlabel('Number of real candidates (+ none)');ax.set_ylabel('Wrong offered answer when correct intent absent (%)');ax.set_ylim(0,100)
        ax.set_title(title_prefix+'Held-out labels: missing-answer failures with lexical hard negatives');ax.legend()
        fig.tight_layout();fig.savefig(root/'missing_answer_by_cardinality.png',dpi=150);plt.close(fig)
    rows=[r for r in report.get('request_benchmark',{}).get('rows',[]) if r['status']=='completed']
    if rows:
        df=pd.DataFrame(rows);fig,ax=plt.subplots(figsize=(9,5))
        for bs,part in df.groupby('candidate_batch_size'):
            a=part.groupby('candidate_count').p50_ms.median();ax.plot(a.index,a.values,marker='o',label=f'candidate batch {bs}')
        ax.set_xlabel('Real candidates in one request');ax.set_ylabel('Complete-request latency (ms)\nMedian of per-message medians')
        ax.set_title(title_prefix+'Full dynamic requests: tokenizer + K encodings + none + JSON (no server)');ax.legend()
        fig.tight_layout();fig.savefig(root/'complete_request_latency.png',dpi=150);plt.close(fig)

Writing phase2d_workflow.py


## 4 · Configure the run

The default reference path already points to your successful Phase 2C archive. `standard` uses 32 numerical-diagnostic examples, with layer traces for four. A quarter of the diagnostic sample is selected for previously high drift; the remainder is sampled from old development data. This is a **targeted debugging sample**, not an unbiased estimate of how often batch changes happen.

`run_fp32=True` attempts a short full-model FP32 diagnostic using the **same model object**, with conservative memory checks. It logs `skipped_memory` or `failed_oom` rather than silently using CPU offload or a quantized model. An A100 provides more headroom; an L4 may pass or skip depending on free memory.

`quick` reduces data and diagnostic samples. `smoke` validates plumbing only. Changing a seed creates different sampling, but does not by itself prove new tests are independent of every past phase.

### Standard dynamic data sizes
| Partition | Unique new messages | Episodes |
|---|---:|---:|
| Train | 500 | 1,000 |
| Development | 150 | 300 |
| Calibration fit | 200 | 400 |
| Calibration validation | 150 | 300 |
| Seen-label test | 100 | 1,600 |
| Held-out-label test | 100 | 1,600 |

Training/development/calibration create a present/absent pair for one sampled candidate count and sampler per message. Each test message receives all four counts × two samplers × two presence conditions. **These are repeated conditions on 100 messages, not 1,600 independent messages.** Confidence intervals resample messages.

In [6]:
from phase2d_common import Settings, json_ready
from phase2d_workflow import Workflow
from dataclasses import asdict
import json

CFG = Settings(
    preset="standard",                  # "quick" or "smoke" for smaller preliminary runs
    run_numerics=True,
    run_fp32=True,                       # memory-guarded; actual skip/failure is reported
    run_none_ablation=True,
    run_request_benchmark=True,
    benchmark_batches=(1, 2, 4),
    benchmark_repeats=5,
    mount_drive=True,
    save_to_drive=True,
    # prior_archive="/content/drive/MyDrive/Colab Notebooks/OpenKind_Phase2C_results/20260917T222948Z/openkind_phase2c_20260917T222948Z.zip",
    # resume_run_id="...",              # local files must still exist; configuration must match
)
W = Workflow(CFG)
print(json.dumps(json_ready({"configuration": asdict(CFG), "sizes": CFG.sizes(), "output": W.out}), indent=2))

{
  "configuration": {
    "preset": "standard",
    "prior_archive": "/content/drive/MyDrive/Colab Notebooks/OpenKind_Phase2C_results/20260917T222948Z/openkind_phase2c_20260917T222948Z.zip",
    "expected_archive_sha256": "2952c2fcca6638c0cc1d1e1060a567b849e46424eb2d16a078d1945233bffb63",
    "model_id": "Qwen/Qwen3.5-4B-Base",
    "model_revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
    "nli_revision": "da70db2af9d09693783c3320c4249840212ee221",
    "banking_revision": "90d4e2ee5521c04fc1488f065b8b083658768c57",
    "output_root": "/content/openkind_phase2d",
    "resume_run_id": "",
    "mount_drive": true,
    "save_to_drive": true,
    "run_numerics": true,
    "run_fp32": true,
    "run_none_ablation": true,
    "run_request_benchmark": true,
    "diagnostic_seed": 17041,
    "data_seed": 17043,
    "max_length": 256,
    "fit_candidate_counts": [
      2,
      4,
      8,
      16
    ],
    "eval_candidate_counts": [
      2,
      4,
      8,
      16
    ],
    "sam

## 5 · Authenticate, verify the archive, and replay saved head fixtures

`HF_TOKEN` is read from Colab Secrets, falling back to the environment. It is never printed or exported. Read access is enough for the public model/datasets.

This checks the exact archive SHA-256, source run, label order, dynamic prompt contract, and saved CPU head fixtures. A missing archive or checksum mismatch stops the run. Use the specified run rather than pointing at an arbitrary ZIP with the same filename.

In [7]:
W.setup()

MessageError: Error: credential propagation was unsuccessful

## 6 · Load one text-only model

The loader is the same text-only causal-model path used successfully in Phase 2B/2C. After validation, it retains the text backbone and discards the generation wrapper. It does not generate tokens, load vision weights, or instantiate a second copy.

A small replay of the old token fixtures is logged separately from the new diagnostics. Replay differences are not silently erased by retraining the head.

In [ ]:
W.load_model()

## 7 · Construct the fresh dynamic-choice experiment

All new dynamic partitions exclude normalized messages appearing anywhere in Phase 2C. Tests use unused messages from the original Banking77 test source; fitting partitions use unused training-source messages. Source CSV contents are hash-checked against pinned author metadata; no legacy dataset script executes.

**Uniform distractors** are sampled from the allowed label pool. **Lexical hard distractors** are selected by similarity of label descriptions, with separate TF-IDF vocabularies for seen and held-out pools. The true label is used *offline* to construct the experiment. This is not a production retriever or evidence that these are the model's hardest negatives.

For each message/sampler, negative sets are nested as K increases. Paired episodes either include the annotated intent or replace it with another distractor. None means “the annotated intent was omitted”—not arbitrary uncertainty or a safety judgement.

Repeated runs of this notebook's `data_seed` reuse the same new test messages. The model's pretraining contamination is unknown.

In [ ]:
W.load_data()

## 8 · Trace execution-shape differences under each precision configuration

For **each configuration**, compare the same input alone against repeated-alone, duplicate-batch, right-padding, mixed-length, and left-padding/explicit-position execution. Record maximum probability shifts, selected-class changes, threshold crossings, and last-token representation differences.

The trace copies only one final valid token per selected module to the CPU: embedding, each decoder block, and final norm. It does not retain all tokens or all layers for the entire dataset. Between-precision single-call shifts are recorded separately from within-precision batch shifts.

The mode names describe requested controls. Actual parameter dtypes and arithmetic flags are recorded. Hooks and precision settings are removed/restored after each mode. A failed mode cannot be reported as stable.

In [ ]:
W.numerical_diagnostics()

## 9 · Cache frozen candidate scores once

Candidate scoring uses **BF16/default SDPA, one unpadded input per call**, regardless of the diagnostic verdict above. The scalar scores are cached in SQLite; prompt tokens, checkpoint/head hashes, code, software/hardware, and backend settings form the cache identity.

The scorer and Qwen remain frozen. Repeated present/absent/count episodes reuse identical candidate scores. This saves repeated offline computation without claiming shared-state inference. Full-request timings later deliberately bypass this cache.

Local cache commits occur every 50 new scores. An interrupted cell can reuse committed scores while the local runtime files survive. The cache is not uploaded automatically and does not survive runtime deletion.

In [ ]:
W.score()

## 10 · Fit only the missing-answer models

Compare:

| Model | New trainable coefficients | Meaning |
|---|---:|---|
| `frozen_global` | 0 | Original Phase 2C none logit |
| `refit_global` | 1 | Refit only the constant on new training data |
| `count_bias` | 2 | Constant + coefficient on log(K) |
| `set_linear` | 7 | Constant + standardized, permutation-invariant score features |

Set features are maximum score, top-two gap, mean, population standard deviation, log-mean-exp, and log(K). The real candidate logits do not change.

**Control the omitted-intent prior:** raw paired episodes are 50% absent. Fitting, development selection, temperature fitting, and calibration validation reweight each message to an **assumed 25% absent prior**, matching the intended earlier sampling regime rather than simply teaching the model to reject half of requests. This is an experimental assumption, not a measured deployment prior.

Select the none architecture by development weighted NLL only. Fit temperature on calibration-fit messages; apply it only if a different calibration-validation set improves by at least 0.002 weighted NLL. No test metrics select a model or temperature. The previous calibrated temperature is not reused for a newly fitted rejection head.

In [ ]:
W.fit_rejection()

## 11 · Evaluate the chosen policy and its tradeoffs

Report both overall scores and the crucial conditional errors: **wrong offered answer when the correct intent is absent**, false abstention when the correct intent is present, and wrong candidate selection when answerable. Include candidate-count/sampler breakdowns, raw versus temperature-scaled probabilities, and weighted-NLL scenarios for assumed absent priors 5%, 25%, and 50%.

The prior frozen none logit remains an explicit control. Model comparisons use the same test episodes and message-clustered bootstrap intervals. Uniform and lexical-hard test conditions remain separate. A reduction in wrong offered answers is not automatically a success if it comes from excessive rejection of answerable requests.

The larger test episode count does not create new independent messages. No pooled ECE value is treated as a guarantee of safe decisions.

In [ ]:
W.evaluate()

## 12 · Time complete requests, not isolated candidate passes

Use the same small set of development messages at K=2,4,8,16, evaluated with candidate batch sizes 1,2,4. The timed function includes segmented tokenization, device transfers, **all K backbone encodings**, candidate logits, selected none model, softmax, CPU synchronization, and JSON assembly. There is no server/network or generated text.

Warmup calls are excluded. Record actual input lengths, total tokens processed, model-call counts, peak extra allocation, and output differences versus unpadded batch-one inference. A faster batch that changes probabilities or choices is flagged, not silently accepted. These five-repeat p95 values are descriptive, not production-tail estimates.

This is the complete-request baseline for a later shared-prefix implementation; it does not implement that optimization.

In [ ]:
W.benchmark()

## 13 · Export a reproducible handoff

Export the unchanged candidate head, each none model, selected model/temperature manifest, numerical feature definitions, and development-set golden score→none→probability vectors. Preserve model, archive, and code identities.

The export contains no 4B backbone weights or credentials. It supports a later Rust postprocessing parity test; it is not itself evidence of Rust/Metal or Jev API compatibility.

In [ ]:
W.export()

## 14 · Write reports, archive, and copy to Drive

The final summary states completed, skipped, disabled, and failed stages. Read the **numerical verdicts**, conditional missing-answer errors, and per-request batching differences together before making deployment decisions.

Outputs go to a new local run directory and, when enabled, a new folder below:
`MyDrive/Colab Notebooks/OpenKind_Phase2D_results/<run_id>/`.

Your Phase 2B/2C files are never modified. If an earlier stage failed, `W.finalize()` can still save a partial report; a partial archive is not a completed benchmark.

In [ ]:
archive = W.finalize()
from IPython.display import FileLink, display
for p in (W.out / "paste_back_summary.json", W.out / "openkind_phase2d_summary.json", archive):
    display(FileLink(str(p)))
print("Send paste_back_summary.json or the BEGIN/END summary block. No HF_TOKEN is included.")

## 15 · Inspect the four report figures

Figures contain this run's measured data only. Missing figures mean the relevant stage was disabled, skipped, or did not complete; inspect the report instead of assuming a zero result.

In [ ]:
from IPython.display import Image, display
for name in ("precision_shape_drift.png", "layer_drift.png",
             "missing_answer_by_cardinality.png", "complete_request_latency.png"):
    path = W.out / "plots" / name
    if path.exists():
        display(Image(filename=str(path)))

## Sources and interpretation boundaries

### Prior work used as the starting point
- Saved Phase 2C notebook: https://colab.research.google.com/drive/1iYTfdtn1-G26zpNyK98iodp_IGBi1YAz
- Actual run: `20260917T222948Z`; archive SHA-256 `2952c2fcca6638c0cc1d1e1060a567b849e46424eb2d16a078d1945233bffb63`.
- Both neural heads, their normalization, prompt segments, candidate-label pools, and old-message exclusion groups are read from that run. New rejection heads and diagnostic settings are **Phase 2D experimental additions**, not previously measured improvements.

### Implementation references
- Hugging Face Qwen3.5 API: https://huggingface.co/docs/transformers/model_doc/qwen3_5
- Version-pinned model implementation: https://github.com/huggingface/transformers/blob/v5.17.0/src/transformers/models/qwen3_5/modeling_qwen3_5.py
- PyTorch numerical-accuracy notes: https://docs.pytorch.org/docs/main/notes/numerical_accuracy.html
- SDPA backend control: https://docs.pytorch.org/docs/main/generated/torch.nn.attention.sdpa_kernel.html
- Native BF16 detection: https://docs.pytorch.org/docs/main/generated/torch.cuda.is_bf16_supported.html
- Banking77 original author data: https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data

**Authoring validation:** CPU contracts and a synthetic end-to-end notebook workflow were tested separately, including the actual saved Phase 2C head fixtures. The complete 4B Qwen/CUDA/FP32 experiment and fresh online dataset download were not executed in the authoring environment. Select a native-BF16 GPU and Run all here to perform those checks. Do not treat the separate synthetic validation plots or timings as Qwen results.